In [2]:
import pandas as pd

df = pd.read_csv("economy_technology_miscellenous.csv")

print("SUCCESS")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

SUCCESS
Rows: 133
Columns: ['Question ID', 'Domain', 'Subdomain', 'English Question', 'Nepali Question', 'Gold Answer (English)', 'Gold Answer (Nepali)', 'Explanation', 'Source', 'Source ID', 'Source Title', 'Source URL', 'Source Type', 'Difficulty', 'Answer Type', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17']


In [3]:
import ollama

question = df.iloc[0]["English Question"]

response = ollama.chat(
    model="gemma3:4b",
    messages=[
        {
            "role": "user",
            "content": question
        }
    ]
)

print("Question:", question)
print("Gemma's Answer:", response.message.content)


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [ ]:
gold_answer = df.iloc[0]["Gold Answer (English)"]

print("Question:", question)
print("Gold Answer:", gold_answer)
print("Gemma's Answer:", response.message.content)

Question: What is the official currency of Nepal?
Gold Answer: Nepalese Rupee
Gemma's Answer: The official currency of Nepal is the **Nepalese Rupee (NPR)**. 

It's represented by the code **NPR**.

As of today, November 2, 2023, the exchange rate is approximately:

*   1 USD = 130 NPR
*   1 EUR = 125 NPR

You can always find the most up-to-date exchange rates on websites like Google Finance, XE.com, or a reputable currency converter.


In [ ]:
import ollama
import pandas as pd

results = []

for index, row in df.iterrows():
    question = row["English Question"]

    print(f"Processing {index + 1}/{len(df)}")

    response = ollama.chat(
        model="gemma3:4b",
        messages=[
            {
                "role": "user",
                "content": question
            }
        ]
    )

    results.append({
        "Question ID": row["Question ID"],
        "Question": question,
        "Gold Answer": row["Gold Answer (English)"],
        "Gemma Answer": response.message.content
    })

print("Finished!")

Processing 1/133
Processing 2/133
Processing 3/133
Processing 4/133
Processing 5/133
Processing 6/133
Processing 7/133
Processing 8/133
Processing 9/133
Processing 10/133
Processing 11/133
Processing 12/133
Processing 13/133
Processing 14/133
Processing 15/133
Processing 16/133
Processing 17/133
Processing 18/133
Processing 19/133
Processing 20/133
Processing 21/133
Processing 22/133
Processing 23/133
Processing 24/133
Processing 25/133
Processing 26/133
Processing 27/133
Processing 28/133
Processing 29/133
Processing 30/133
Processing 31/133
Processing 32/133
Processing 33/133
Processing 34/133
Processing 35/133
Processing 36/133
Processing 37/133
Processing 38/133
Processing 39/133
Processing 40/133
Processing 41/133
Processing 42/133
Processing 43/133
Processing 44/133
Processing 45/133
Processing 46/133
Processing 47/133
Processing 48/133
Processing 49/133
Processing 50/133
Processing 51/133
Processing 52/133
Processing 53/133
Processing 54/133
Processing 55/133
Processing 56/133
P

In [ ]:
print("Total results:", len(results))
print("\nFirst result:")
print(results[0])

Total results: 133

First result:
{'Question ID': 'ECO001', 'Question': 'What is the official currency of Nepal?', 'Gold Answer': 'Nepalese Rupee', 'Gemma Answer': "The official currency of Nepal is the **Nepalese Rupee (NPR)**.\n\nIt's represented by the code **NPR** and the symbol **Rs**.\n\nAs of today, November 2, 2023, the exchange rate is roughly 1 USD = 130 NPR.\n\nYou can find the most up-to-date exchange rates on websites like:\n\n*   XE.com: [https://xe.com/currency/npr](https://xe.com/currency/npr)\n*   Google Finance: [https://www.google.com/finance/quote/NPR-NPR](https://www.google.com/finance/quote/NPR-NPR)"}


In [ ]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    "gemma3_4b_raw_results.csv",
    index=False
)

print("Saved successfully!")
print("Rows saved:", len(results_df))
print("Columns:", results_df.columns.tolist())

Saved successfully!
Rows saved: 133
Columns: ['Question ID', 'Question', 'Gold Answer', 'Gemma Answer']


In [ ]:
results_df.head()


,Question ID,Question,Gold Answer,Gemma Answer
0,ECO001,What is the official currency of Nepal?,Nepalese Rupee,The official currency of Nepal is the **Nepale...
1,ECO002,What is the ISO 4217 currency code for the Nep...,NPR,The ISO 4217 currency code for the Nepalese Ru...
2,ECO007,In which year was Nepal Bank Limited established?,1937 (1994 BS),Nepal Bank Limited was established in **1963**...
3,ECO008,Which act currently governs the establishment ...,"Nepal Rastra Bank Act, 2002",The primary act currently governing the establ...
4,ECO009,Which was the first fully state-owned commerci...,Rastriya Banijya Bank,The first fully state-owned commercial bank es...


In [ ]:
# Load the saved raw Gemma results
results_df = pd.read_csv("gemma3_4b_raw_results.csv")

# Combine the original dataset with Gemma's answers
evaluation_df = df.copy()

evaluation_df["Gemma Answer"] = results_df["Gemma Answer"]

print("Rows:", len(evaluation_df))
print("Columns:", evaluation_df.columns.tolist())

Rows: 133
Columns: ['Question ID', 'Domain', 'Subdomain', 'English Question', 'Nepali Question', 'Gold Answer (English)', 'Gold Answer (Nepali)', 'Explanation', 'Source', 'Source ID', 'Source Title', 'Source URL', 'Source Type', 'Difficulty', 'Answer Type', 'Gemma Answer']


In [ ]:
evaluation_df[
    [
        "Question ID",
        "English Question",
        "Gold Answer (English)",
        "Explanation",
        "Gemma Answer"
    ]
].head(3)

,Question ID,English Question,Gold Answer (English),Explanation,Gemma Answer
0,ECO001,What is the official currency of Nepal?,Nepalese Rupee,"Nepal's legal tender is the Nepalese Rupee, is...",The official currency of Nepal is the **Nepale...
1,ECO002,What is the ISO 4217 currency code for the Nep...,NPR,NPR is the internationally recognized three-le...,The ISO 4217 currency code for the Nepalese Ru...
2,ECO007,In which year was Nepal Bank Limited established?,1937 (1994 BS),"Nepal Bank Limited was established in 1937, ma...",Nepal Bank Limited was established in **1963**...


In [ ]:
evaluation_df["Verdict"] = ""
evaluation_df["Error Type"] = ""

print("Rows:", len(evaluation_df))
print("New columns added:")
print(["Verdict", "Error Type"])

Rows: 133
New columns added:
['Verdict', 'Error Type']


In [ ]:
row = evaluation_df.iloc[0]

print("Question:")
print(row["English Question"])

print("\nGold Answer:")
print(row["Gold Answer (English)"])

print("\nExplanation:")
print(row["Explanation"])

print("\nGemma Answer:")
print(row["Gemma Answer"])

Question:
What is the official currency of Nepal?

Gold Answer:
Nepalese Rupee

Explanation:
Nepal's legal tender is the Nepalese Rupee, issued by Nepal Rastra Bank.

Gemma Answer:
The official currency of Nepal is the **Nepalese Rupee (NPR)**.

It's represented by the code **NPR** and the symbol **Rs**.

As of today, November 2, 2023, the exchange rate is roughly 1 USD = 130 NPR.

You can find the most up-to-date exchange rates on websites like:

*   XE.com: [https://xe.com/currency/npr](https://xe.com/currency/npr)
*   Google Finance: [https://www.google.com/finance/quote/NPR-NPR](https://www.google.com/finance/quote/NPR-NPR)


In [ ]:
evaluation_df.loc[0, "Verdict"] = "Error"
evaluation_df.loc[0, "Error Type"] = "Hallucinated / unsupported elaboration"

print("Question ID:", evaluation_df.loc[0, "Question ID"])
print("Verdict:", evaluation_df.loc[0, "Verdict"])
print("Error Type:", evaluation_df.loc[0, "Error Type"])

Question ID: ECO001
Verdict: Error
Error Type: Hallucinated / unsupported elaboration


In [ ]:
row = evaluation_df.iloc[1]

print("Question:")
print(row["English Question"])

print("\nGold Answer:")
print(row["Gold Answer (English)"])

print("\nExplanation:")
print(row["Explanation"])

print("\nGemma Answer:")
print(row["Gemma Answer"])

Question:
What is the ISO 4217 currency code for the Nepalese Rupee?

Gold Answer:
NPR

Explanation:
NPR is the internationally recognized three-letter currency code for Nepal.

Gemma Answer:
The ISO 4217 currency code for the Nepalese Rupee was **NPR**.

However, it's crucial to note that **the Nepalese Rupee is no longer legal tender.** It was officially replaced by the Nepalese Rupee (NPR) in 2008.  While the code still exists, it's primarily used for historical or statistical purposes.

Currently, Nepal uses the Indian Rupee (INR) as its primary currency.


In [ ]:
evaluation_df.loc[1, "Verdict"] = "Error"
evaluation_df.loc[1, "Error Type"] = "Incorrect factual / unsupported claim"

print("Question ID:", evaluation_df.loc[1, "Question ID"])
print("Verdict:", evaluation_df.loc[1, "Verdict"])
print("Error Type:", evaluation_df.loc[1, "Error Type"])

Question ID: ECO002
Verdict: Error
Error Type: Incorrect factual / unsupported claim


In [ ]:
row = evaluation_df.iloc[2]

print("Question:")
print(row["English Question"])

print("\nGold Answer:")
print(row["Gold Answer (English)"])

print("\nExplanation:")
print(row["Explanation"])

print("\nGemma Answer:")
print(row["Gemma Answer"])

Question:
In which year was Nepal Bank Limited established?

Gold Answer:
1937 (1994 BS)

Explanation:
Nepal Bank Limited was established in 1937, marking the start of modern banking in Nepal.

Gemma Answer:
Nepal Bank Limited was established in **1963**. 

It began operations on December 14, 1963, with an initial capital of Rs. 1.5 crore.


In [ ]:
evaluation_df.loc[2, "Verdict"] = "Error"
evaluation_df.loc[2, "Error Type"] = "Incorrect factual / unsupported claim"

print("Question ID:", evaluation_df.loc[2, "Question ID"])
print("Verdict:", evaluation_df.loc[2, "Verdict"])
print("Error Type:", evaluation_df.loc[2, "Error Type"])

Question ID: ECO007
Verdict: Error
Error Type: Incorrect factual / unsupported claim


In [ ]:
df["Nepali Question"]

0                         नेपालको आधिकारिक मुद्रा के हो?
1            नेपाली रुपैयाँको ISO 4217 मुद्रा कोड के हो?
2        नेपाल बैंक लिमिटेड कुन सालमा स्थापना भएको थियो?
3      हाल नेपाल राष्ट्र बैंकको स्थापना र कार्यलाई कु...
4      नेपाल बैंक लिमिटेड पछि स्थापना भएको पहिलो पूर्...
                             ...                        
128    राष्ट्रिय जनगणना तथ्याङ्क अनुसार नेपालभर लगभग ...
129    नेपालको सरकारी प्रशासनमा प्रयोग हुने आधिकारिक ...
130    नेपाललाई प्रायः बहु-जातीय, बहु-भाषिक, र यसका व...
131    नेपालको जनसंख्याको सबैभन्दा ठूलो हिस्साले कुन ...
132      नेपाली भाषा परम्परागत रूपमा कुन लिपिमा लेखिन्छ?
Name: Nepali Question, Length: 133, dtype: object

In [ ]:
import ollama

nepali_results = []

for index, row in df.iterrows():
    question = row["Nepali Question"]

    print(f"Processing Nepali {index + 1}/{len(df)}")

    response = ollama.chat(
        model="gemma3:4b",
        messages=[
            {
                "role": "user",
                "content": question
            }
        ]
    )

    nepali_results.append({
        "Question ID": row["Question ID"],
        "Nepali Question": question,
        "Gold Answer (Nepali)": row["Gold Answer (Nepali)"],
        "Gemma Answer (Nepali)": response.message.content
    })

print("\nFinished!")
print("Total Nepali results:", len(nepali_results))

Processing Nepali 1/133
Processing Nepali 2/133
Processing Nepali 3/133
Processing Nepali 4/133
Processing Nepali 5/133
Processing Nepali 6/133
Processing Nepali 7/133
Processing Nepali 8/133
Processing Nepali 9/133
Processing Nepali 10/133
Processing Nepali 11/133
Processing Nepali 12/133
Processing Nepali 13/133
Processing Nepali 14/133
Processing Nepali 15/133
Processing Nepali 16/133
Processing Nepali 17/133
Processing Nepali 18/133
Processing Nepali 19/133
Processing Nepali 20/133
Processing Nepali 21/133
Processing Nepali 22/133
Processing Nepali 23/133
Processing Nepali 24/133
Processing Nepali 25/133
Processing Nepali 26/133
Processing Nepali 27/133
Processing Nepali 28/133
Processing Nepali 29/133
Processing Nepali 30/133
Processing Nepali 31/133
Processing Nepali 32/133
Processing Nepali 33/133
Processing Nepali 34/133
Processing Nepali 35/133
Processing Nepali 36/133
Processing Nepali 37/133
Processing Nepali 38/133
Processing Nepali 39/133
Processing Nepali 40/133
Processin

In [ ]:
nepali_results_df = pd.DataFrame(nepali_results)

nepali_results_df.to_csv(
    "gemma3_4b_raw_results_nepali.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully!")
print("Rows saved:", len(nepali_results_df))
print("Columns:", nepali_results_df.columns.tolist())

Saved successfully!
Rows saved: 133
Columns: ['Question ID', 'Nepali Question', 'Gold Answer (Nepali)', 'Gemma Answer (Nepali)']


In [ ]:
# Load both raw result files
english_results = pd.read_csv("gemma3_4b_raw_results.csv")
nepali_results = pd.read_csv("gemma3_4b_raw_results_nepali.csv")

# Create English rows
english_df = pd.DataFrame({
    "Question ID": english_results["Question ID"],
    "Language": "English",
    "Question": english_results["Question"],
    "Gold Answer": english_results["Gold Answer"],
    "Gemma Answer": english_results["Gemma Answer"]
})

# Create Nepali rows
nepali_df = pd.DataFrame({
    "Question ID": nepali_results["Question ID"],
    "Language": "Nepali",
    "Question": nepali_results["Nepali Question"],
    "Gold Answer": nepali_results["Gold Answer (Nepali)"],
    "Gemma Answer": nepali_results["Gemma Answer (Nepali)"]
})

# Combine English + Nepali
all_results_df = pd.concat(
    [english_df, nepali_df],
    ignore_index=True
)

print("Total responses:", len(all_results_df))
print("\nResponses by language:")
print(all_results_df["Language"].value_counts())

Total responses: 266

Responses by language:
Language
English    133
Nepali     133
Name: count, dtype: int64


In [ ]:
all_results_df.to_csv(
    "gemma3_4b_raw_results_bilingual.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully!")
print("Rows:", len(all_results_df))
print("English:", (all_results_df["Language"] == "English").sum())
print("Nepali:", (all_results_df["Language"] == "Nepali").sum())

Saved successfully!
Rows: 266
English: 133
Nepali: 133


In [ ]:
# Load the master bilingual raw results
all_results_df = pd.read_csv(
    "gemma3_4b_raw_results_bilingual.csv"
)

# Create a separate dataframe for evaluation
evaluation_df = all_results_df.copy()

# Add empty evaluation fields
evaluation_df["Verdict"] = ""
evaluation_df["Error Type"] = ""

print("Total rows:", len(evaluation_df))
print("\nRows by language:")
print(evaluation_df["Language"].value_counts())

Total rows: 266

Rows by language:
Language
English    133
Nepali     133
Name: count, dtype: int64


In [ ]:
# Inspect one English and one Nepali example
print("===== ENGLISH =====")
print(evaluation_df[evaluation_df["Language"] == "English"].iloc[0])

print("\n===== NEPALI =====")
print(evaluation_df[evaluation_df["Language"] == "Nepali"].iloc[0])


===== ENGLISH =====
Question ID                                                ECO001
Language                                                  English
Question                  What is the official currency of Nepal?
Gold Answer                                        Nepalese Rupee
Gemma Answer    The official currency of Nepal is the **Nepale...
Verdict                                                          
Error Type                                                       
Name: 0, dtype: object

===== NEPALI =====
Question ID                                                ECO001
Language                                                   Nepali
Question                           नेपालको आधिकारिक मुद्रा के हो?
Gold Answer                                        नेपाली रुपैयाँ
Gemma Answer    नेपालको आधिकारिक मुद्रा रुपैयाँ (Nepali Rupees...
Verdict                                                          
Error Type                                                       
Name: 133, dt

In [ ]:
import ollama

models = ollama.list()

print("Models available in Ollama:\n")

for model in models.models:
    print(model.model)

Models available in Ollama:

gemma3:4b
gemma3:27b
qwen2.5-coder:7b


In [ ]:
import ollama

row = evaluation_df[
    (evaluation_df["Question ID"] == "ECO001") &
    (evaluation_df["Language"] == "English")
].iloc[0]

evaluation_prompt = f"""
You are evaluating an answer produced by an AI model.

Question:
{row["Question"]}

Gold Answer:
{row["Gold Answer"]}

AI Answer:
{row["Gemma Answer"]}

Evaluate the AI answer against the Gold Answer.

Use exactly one of these Verdict values:
- Correct
- Error

If the Verdict is Error, use exactly one of these Error Type values:
- Hallucinated / unsupported elaboration
- Incorrect factual / unsupported claim
- No answer / unsupported-context refusal
- Question repetition / non-answer

If the answer is Correct, use:
Error Type: None

Important:
- Do not require the AI answer to exactly match the wording of the Gold Answer.
- Judge whether the core answer is correct.
- Extra information is an error only when it is unsupported or factually incorrect.
- If the answer contains a factual claim that contradicts the Gold Answer or its intended meaning, classify it as Incorrect factual / unsupported claim.
- Return ONLY this format:

Verdict: <Correct or Error>
Error Type: <one allowed value>
"""

response = ollama.chat(
    model="gemma3:27b",
    messages=[
        {
            "role": "user",
            "content": evaluation_prompt
        }
    ]
)

print(response.message.content)

ResponseError: model requires more system memory (18.4 GiB) than is available (3.3 GiB) (status code: 500)

In [ ]:
import ollama

response = ollama.chat(
    model="qwen2.5-coder:7b",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: TEST SUCCESS"
        }
    ]
)

print(response.message.content)

ResponseError: "qwen2.5-coder:7b" does not support chat (status code: 400)

In [ ]:
import pandas as pd
import ollama
import os
import time

# ============================================================
# SETTINGS
# ============================================================

EXCEL_FILE = r"E:\Desktop\Research_Hallucination\Hexagon-Dataset.xlsx"
MODEL = "gemma3:4b"

# Only the Q&A datasets for this pipeline
QA_SHEETS = [
    "Constitution&Law",
    "Culture+Religion+Literature",
    "Geography and tourism",
    "Education health",
    "Economy + Technology + Miscella"
]

OUTPUT_FOLDER = r"E:\Desktop\Research_Hallucination\gemma_results"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("Setup complete.")
print("Model:", MODEL)
print("Output folder:", OUTPUT_FOLDER)

Setup complete.
Model: gemma3:4b
Output folder: E:\Desktop\Research_Hallucination\gemma_results


In [ ]:
import pandas as pd
import ollama
import os
import time

# ============================================================
# SETTINGS
# ============================================================

EXCEL_FILE = r"E:\Desktop\Research_Hallucination\Hexagon-Dataset.xlsx"
MODEL = "gemma3:4b"

QA_SHEETS = [
    "Constitution&Law",
    "Culture+Religion+Literature",
    "Geography and tourism",
    "Education health",
    "Economy + Technology + Miscella"
]

OUTPUT_FOLDER = r"E:\Desktop\Research_Hallucination\gemma_results"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# ============================================================
# COLUMN NORMALIZATION
# ============================================================

def get_column(df, possible_names):
    for name in possible_names:
        if name in df.columns:
            return name
    raise ValueError(
        f"Could not find any of these columns: {possible_names}"
    )


# ============================================================
# GEMMA FUNCTION
# ============================================================

def ask_gemma(question, language):
    
    if language == "English":
        prompt = f"""Answer the following question accurately and concisely.

Question: {question}

Give only the answer. Do not add explanations, dates, sources, or extra information unless necessary to answer the question."""
    
    else:
        prompt = f"""यस प्रश्नको सही र संक्षिप्त उत्तर दिनुहोस्।

प्रश्न: {question}

केवल उत्तर दिनुहोस्। आवश्यक नभएसम्म व्याख्या, स्रोत वा अतिरिक्त जानकारी नदिनुहोस्।"""

    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.message.content.strip()


# ============================================================
# PROCESS EACH DATASET
# ============================================================

for sheet_name in QA_SHEETS:

    print("\n" + "=" * 70)
    print("PROCESSING:", sheet_name)
    print("=" * 70)

    # Read sheet
    df_sheet = pd.read_excel(
        EXCEL_FILE,
        sheet_name=sheet_name
    )

    # Find correct columns
    id_col = get_column(
        df_sheet,
        ["Question ID"]
    )

    eng_q_col = get_column(
        df_sheet,
        ["English Question"]
    )

    nep_q_col = get_column(
        df_sheet,
        ["Nepali Question"]
    )

    eng_gold_col = get_column(
        df_sheet,
        ["Gold Answer (EN)", "Gold Answer (English)"]
    )

    nep_gold_col = get_column(
        df_sheet,
        ["Gold Answer (NE)", "Gold Answer (Nepali)"]
    )

    # Output filename
    safe_name = "".join(
        c if c.isalnum() or c in (" ", "_", "-") else "_"
        for c in sheet_name
    ).strip()

    output_file = os.path.join(
        OUTPUT_FOLDER,
        f"{safe_name}_gemma_results.csv"
    )

    # Resume existing results if available
    if os.path.exists(output_file):

        results_df = pd.read_csv(output_file)

        completed_ids = set(
            results_df["Question ID"].astype(str)
        )

        print(
            f"Existing results found: {len(results_df)}"
        )

    else:

        results_df = pd.DataFrame(
            columns=[
                "Question ID",
                "Language",
                "Question",
                "Gold Answer",
                "Gemma Answer"
            ]
        )

        completed_ids = set()

    # --------------------------------------------------------
    # PROCESS ROWS
    # --------------------------------------------------------

    for index, row in df_sheet.iterrows():

        question_id = str(row[id_col])

        # Skip rows already processed in a previous run
        # We use ID + language so English and Nepali are separate.
        english_key = question_id + "_English"
        nepali_key = question_id + "_Nepali"

        existing_keys = set(
            results_df["Question ID"].astype(str)
            + "_"
            + results_df["Language"].astype(str)
        )

        # ---------------- ENGLISH ----------------

        if english_key not in existing_keys:

            question = str(row[eng_q_col])
            gold_answer = str(row[eng_gold_col])

            try:

                gemma_answer = ask_gemma(
                    question,
                    "English"
                )

                new_row = {
                    "Question ID": question_id,
                    "Language": "English",
                    "Question": question,
                    "Gold Answer": gold_answer,
                    "Gemma Answer": gemma_answer
                }

                results_df.loc[len(results_df)] = new_row

                # Save immediately
                results_df.to_csv(
                    output_file,
                    index=False,
                    encoding="utf-8-sig"
                )

                print(
                    f"[{index+1}/{len(df_sheet)}] "
                    f"{question_id} English ✓"
                )

            except Exception as e:

                print(
                    f"ERROR {question_id} English: {e}"
                )

                time.sleep(2)

        # ---------------- NEPALI ----------------

        if nepali_key not in existing_keys:

            question = str(row[nep_q_col])
            gold_answer = str(row[nep_gold_col])

            try:

                gemma_answer = ask_gemma(
                    question,
                    "Nepali"
                )

                new_row = {
                    "Question ID": question_id,
                    "Language": "Nepali",
                    "Question": question,
                    "Gold Answer": gold_answer,
                    "Gemma Answer": gemma_answer
                }

                results_df.loc[len(results_df)] = new_row

                # Save immediately
                results_df.to_csv(
                    output_file,
                    index=False,
                    encoding="utf-8-sig"
                )

                print(
                    f"[{index+1}/{len(df_sheet)}] "
                    f"{question_id} Nepali ✓"
                )

            except Exception as e:

                print(
                    f"ERROR {question_id} Nepali: {e}"
                )

                time.sleep(2)

    print("\nFINISHED:", sheet_name)
    print("Results:", output_file)

print("\n" + "=" * 70)
print("ALL Q&A DATASETS FINISHED")
print("=" * 70)


PROCESSING: Constitution&Law
Existing results found: 1809
[1/917] CL001 English ✓

FINISHED: Constitution&Law
Results: E:\Desktop\Research_Hallucination\gemma_results\Constitution_Law_gemma_results.csv

PROCESSING: Culture+Religion+Literature
Existing results found: 848

FINISHED: Culture+Religion+Literature
Results: E:\Desktop\Research_Hallucination\gemma_results\Culture_Religion_Literature_gemma_results.csv

PROCESSING: Geography and tourism


KeyError: 'Question ID'

In [ ]:
import os
import pandas as pd

folder = r"E:\Desktop\Research_Hallucination\gemma_results"

print("Files currently saved:\n")

for filename in os.listdir(folder):
    path = os.path.join(folder, filename)

    if filename.lower().endswith(".csv"):
        try:
            temp = pd.read_csv(path)

            print(
                f"{filename} → "
                f"{len(temp)} rows → "
                f"columns: {temp.columns.tolist()}"
            )

        except Exception as e:
            print(f"{filename} → ERROR: {e}")

Files currently saved:

Constitution_Law_gemma_results.csv → 1810 rows → columns: ['Question ID', 'Language', 'Question', 'Gold Answer', 'Gemma Answer']
Culture_Religion_Literature_gemma_results.csv → 848 rows → columns: ['Question ID', 'Language', 'Question', 'Gold Answer', 'Gemma Answer']
Geography and tourism_gemma_results.csv → 0 rows → columns: ['Unnamed: 0']


In [ ]:
import pandas as pd
import ollama
import os
import time

EXCEL_FILE = r"E:\Desktop\Research_Hallucination\Hexagon-Dataset.xlsx"
MODEL = "gemma3:4b"

QA_SHEETS = [
    "Constitution&Law",
    "Culture+Religion+Literature",
    "Geography and tourism",
    "Education health",
    "Economy + Technology + Miscella"
]

OUTPUT_FOLDER = r"E:\Desktop\Research_Hallucination\gemma_results"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def get_column(df, names):
    for name in names:
        if name in df.columns:
            return name
    raise ValueError(f"Column not found: {names}")


def ask_gemma(question, language):

    if language == "English":
        prompt = f"""Answer this question accurately and concisely.

Question: {question}

Give only the answer. Do not add unrelated information."""
    else:
        prompt = f"""यस प्रश्नको सही र संक्षिप्त उत्तर दिनुहोस्।

प्रश्न: {question}

केवल उत्तर दिनुहोस्। अनावश्यक जानकारी नदिनुहोस्।"""

    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.message.content.strip()


for sheet_name in QA_SHEETS:

    print("\n" + "=" * 70)
    print("PROCESSING:", sheet_name)
    print("=" * 70)

    df_sheet = pd.read_excel(
        EXCEL_FILE,
        sheet_name=sheet_name
    )

    id_col = get_column(df_sheet, ["Question ID"])
    eng_q_col = get_column(df_sheet, ["English Question"])
    nep_q_col = get_column(df_sheet, ["Nepali Question"])

    eng_gold_col = get_column(
        df_sheet,
        ["Gold Answer (EN)", "Gold Answer (English)"]
    )

    nep_gold_col = get_column(
        df_sheet,
        ["Gold Answer (NE)", "Gold Answer (Nepali)"]
    )

    safe_name = "".join(
        c if c.isalnum() or c in (" ", "_", "-") else "_"
        for c in sheet_name
    ).strip()

    output_file = os.path.join(
        OUTPUT_FOLDER,
        f"{safe_name}_gemma_results.csv"
    )

    # --------------------------------------------------------
    # LOAD EXISTING RESULTS SAFELY
    # --------------------------------------------------------

    required_columns = [
        "Question ID",
        "Language",
        "Question",
        "Gold Answer",
        "Gemma Answer"
    ]

    if os.path.exists(output_file):

        try:
            existing = pd.read_csv(
                output_file,
                encoding="utf-8-sig"
            )

            if all(col in existing.columns for col in required_columns):

                results_df = existing[required_columns].copy()

                print(
                    "Existing valid results:",
                    len(results_df)
                )

            else:

                print(
                    "Existing file is invalid/empty. "
                    "Starting this dataset again."
                )

                results_df = pd.DataFrame(
                    columns=required_columns
                )

        except Exception as e:

            print(
                "Could not read existing results:",
                e
            )

            results_df = pd.DataFrame(
                columns=required_columns
            )

    else:

        results_df = pd.DataFrame(
            columns=required_columns
        )

    # Create completed keys
    completed_keys = set(
        results_df["Question ID"].astype(str)
        + "_"
        + results_df["Language"].astype(str)
    )

    # --------------------------------------------------------
    # PROCESS QUESTIONS
    # --------------------------------------------------------

    for index, row in df_sheet.iterrows():

        question_id = str(row[id_col])

        # ====================================================
        # ENGLISH
        # ====================================================

        key = question_id + "_English"

        if key not in completed_keys:

            question = str(row[eng_q_col])
            gold_answer = str(row[eng_gold_col])

            try:

                answer = ask_gemma(
                    question,
                    "English"
                )

                new_row = pd.DataFrame([{
                    "Question ID": question_id,
                    "Language": "English",
                    "Question": question,
                    "Gold Answer": gold_answer,
                    "Gemma Answer": answer
                }])

                results_df = pd.concat(
                    [results_df, new_row],
                    ignore_index=True
                )

                completed_keys.add(key)

                results_df.to_csv(
                    output_file,
                    index=False,
                    encoding="utf-8-sig"
                )

                print(
                    f"[{index+1}/{len(df_sheet)}] "
                    f"{question_id} English ✓"
                )

            except Exception as e:

                print(
                    f"ERROR {question_id} English: {e}"
                )

                time.sleep(2)

        # ====================================================
        # NEPALI
        # ====================================================

        key = question_id + "_Nepali"

        if key not in completed_keys:

            question = str(row[nep_q_col])
            gold_answer = str(row[nep_gold_col])

            try:

                answer = ask_gemma(
                    question,
                    "Nepali"
                )

                new_row = pd.DataFrame([{
                    "Question ID": question_id,
                    "Language": "Nepali",
                    "Question": question,
                    "Gold Answer": gold_answer,
                    "Gemma Answer": answer
                }])

                results_df = pd.concat(
                    [results_df, new_row],
                    ignore_index=True
                )

                completed_keys.add(key)

                results_df.to_csv(
                    output_file,
                    index=False,
                    encoding="utf-8-sig"
                )

                print(
                    f"[{index+1}/{len(df_sheet)}] "
                    f"{question_id} Nepali ✓"
                )

            except Exception as e:

                print(
                    f"ERROR {question_id} Nepali: {e}"
                )

                time.sleep(2)

    print(
        "\nFINISHED:",
        sheet_name,
        "| Total saved:",
        len(results_df)
    )


print("\n" + "=" * 70)
print("ALL Q&A DATASETS FINISHED")
print("=" * 70)


PROCESSING: Constitution&Law
Existing valid results: 1810

FINISHED: Constitution&Law | Total saved: 1810

PROCESSING: Culture+Religion+Literature
Existing valid results: 848

FINISHED: Culture+Religion+Literature | Total saved: 848

PROCESSING: Geography and tourism
Existing file is invalid/empty. Starting this dataset again.
[1/234] GEO001 English ✓
[1/234] GEO001 Nepali ✓
[2/234] GEO002 English ✓
[2/234] GEO002 Nepali ✓
[3/234] GEO003 English ✓
[3/234] GEO003 Nepali ✓
[4/234] GEO004 English ✓
[4/234] GEO004 Nepali ✓
[5/234] GEO005 English ✓
[5/234] GEO005 Nepali ✓
[6/234] TOU001 English ✓
[6/234] TOU001 Nepali ✓
[7/234] TOU002 English ✓
[7/234] TOU002 Nepali ✓
[8/234] TOU003 English ✓
[8/234] TOU003 Nepali ✓
[9/234] TOU004 English ✓
[9/234] TOU004 Nepali ✓
[10/234] GEO006 English ✓
[10/234] GEO006 Nepali ✓
[11/234] TOU005 English ✓
[11/234] TOU005 Nepali ✓
[12/234] GEO007 English ✓
[12/234] GEO007 Nepali ✓
[13/234] TOU006 English ✓
[13/234] TOU006 Nepali ✓
[14/234] GEO008 English ✓

In [ ]:
import os
import pandas as pd

folder = r"E:\Desktop\Research_Hallucination\gemma_results"

print("=" * 70)
print("FINAL GEMMA RESULT FILES")
print("=" * 70)

total = 0

for filename in sorted(os.listdir(folder)):

    if filename.endswith(".csv"):

        path = os.path.join(folder, filename)
        data = pd.read_csv(path)

        print(f"{filename}: {len(data)} rows")

        total += len(data)

print("=" * 70)
print("TOTAL GENERATED ANSWERS:", total)
print("=" * 70)

FINAL GEMMA RESULT FILES
Constitution_Law_gemma_results.csv: 1810 rows
Culture_Religion_Literature_gemma_results.csv: 848 rows
Economy _ Technology _ Miscella_gemma_results.csv: 266 rows
Education health_gemma_results.csv: 316 rows
Geography and tourism_gemma_results.csv: 466 rows
TOTAL GENERATED ANSWERS: 3706


In [ ]:
import pandas as pd
import os

EXCEL_FILE = r"E:\Desktop\Research_Hallucination\Hexagon-Dataset.xlsx"
FOLDER = r"E:\Desktop\Research_Hallucination\gemma_results"

datasets = {
    "Constitution&Law": "Constitution_Law_gemma_results.csv",
    "Culture+Religion+Literature": "Culture_Religion_Literature_gemma_results.csv",
    "Economy + Technology + Miscella": "Economy _ Technology _ Miscella_gemma_results.csv",
    "Education health": "Education health_gemma_results.csv",
    "Geography and tourism": "Geography and tourism_gemma_results.csv"
}

print("=" * 70)
print("FINDING MISSING ANSWERS")
print("=" * 70)

missing_total = 0

for sheet_name, filename in datasets.items():

    original = pd.read_excel(
        EXCEL_FILE,
        sheet_name=sheet_name
    )

    results = pd.read_csv(
        os.path.join(FOLDER, filename)
    )

    id_col = "Question ID"

    completed = set(
        results["Question ID"].astype(str)
        + "_"
        + results["Language"].astype(str)
    )

    missing = []

    for _, row in original.iterrows():

        qid = str(row[id_col])

        if qid + "_English" not in completed:
            missing.append(qid + " — English")

        if qid + "_Nepali" not in completed:
            missing.append(qid + " — Nepali")

    print(f"\n{sheet_name}")
    print("Missing:", len(missing))

    if missing:
        print(missing)

    missing_total += len(missing)

print("\n" + "=" * 70)
print("TOTAL MISSING:", missing_total)
print("=" * 70)

FINDING MISSING ANSWERS

Constitution&Law
Missing: 0

Culture+Religion+Literature
Missing: 0

Economy + Technology + Miscella
Missing: 0

Education health
Missing: 0

Geography and tourism
Missing: 0

TOTAL MISSING: 0


In [ ]:
import pandas as pd
import ollama

EXCEL_FILE = r"E:\Desktop\Research_Hallucination\Hexagon-Dataset.xlsx"
MODEL = "gemma3:4b"

poly_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Poly-FEVER"
)

print("Poly-FEVER rows:", len(poly_df))
print("Columns:", poly_df.columns.tolist())

def classify_statement(statement, language):

    if language == "English":
        prompt = f"""Decide whether this statement is TRUE or FALSE.

Statement:
{statement}

Reply with exactly one word:
TRUE
or
FALSE"""

    else:
        prompt = f"""यो कथन TRUE हो कि FALSE हो?

कथन:
{statement}

ठ्याक्कै एउटा शब्द मात्र लेख्नुहोस्:
TRUE
वा
FALSE"""

    response = ollama.chat(
        model="gemma3:4b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "num_ctx": 512,
            "temperature": 0
        },
        keep_alive=0
    )

    return response.message.content.strip()

# Test first 5 English + 5 Nepali
test_results = []

for _, row in poly_df.head(5).iterrows():

    test_results.append({
        "ID": row["ID"],
        "Language": "English",
        "Statement": row["en"],
        "Gold Label": row["Label"],
        "Gemma Prediction": classify_statement(
            str(row["en"]),
            "English"
        )
    })

    test_results.append({
        "ID": row["ID"],
        "Language": "Nepali",
        "Statement": row["Nepali"],
        "Gold Label": row["Label"],
        "Gemma Prediction": classify_statement(
            str(row["Nepali"]),
            "Nepali"
        )
    })

test_df = pd.DataFrame(test_results)

display(test_df)

Poly-FEVER rows: 11999
Columns: ['ID', 'Label', 'Topic Distribution', 'en', 'Nepali', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17']


ResponseError: memory layout cannot be allocated (status code: 500)

In [ ]:
test = poly_df.iloc[0]

print("ID:", test["ID"])
print("Gold Label:", test["Label"])

print("\nEnglish statement:")
print(test["en"])

print("\nEnglish prediction:")
print(classify_statement(str(test["en"]), "English"))

print("\nNepali statement:")
print(test["Nepali"])

print("\nNepali prediction:")
print(classify_statement(str(test["Nepali"]), "Nepali"))

ID: 1
Gold Label: True

English statement:
Aramais Yepiskoposyan played for FC Ararat Yerevan, an Armenian football club based in Yerevan during 1986 to 1991.

English prediction:
TRUE

Nepali statement:
अरामाइस येपिस्कोपोस्यानले १९८६ देखि १९९१ सम्म येरेभानमा आधारित आर्मेनियाली फुटबल क्लब एफसी अरारात येरेभानका लागि खेले।

Nepali prediction:
TRUE


In [ ]:
import pandas as pd
import ollama
import os
import time

EXCEL_FILE = r"E:\Desktop\Research_Hallucination\Hexagon-Dataset.xlsx"
OUTPUT_FILE = r"E:\Desktop\Research_Hallucination\polyfever_gemma_results.csv"

MODEL = "gemma3:4b"

# ------------------------------------------------------------
# Load master Poly-FEVER dataset
# ------------------------------------------------------------
poly_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Poly-FEVER"
)

print("Poly-FEVER rows:", len(poly_df))

# ------------------------------------------------------------
# Load existing results if any
# ------------------------------------------------------------
required_columns = [
    "ID",
    "Language",
    "Statement",
    "Gold Label",
    "Gemma Prediction"
]

if os.path.exists(OUTPUT_FILE):

    try:
        results_df = pd.read_csv(OUTPUT_FILE)

        if all(col in results_df.columns for col in required_columns):
            results_df = results_df[required_columns].copy()
        else:
            print("Existing file invalid. Starting fresh.")
            results_df = pd.DataFrame(columns=required_columns)

    except Exception as e:
        print("Could not read existing file:", e)
        results_df = pd.DataFrame(columns=required_columns)

else:
    results_df = pd.DataFrame(columns=required_columns)

print("Existing predictions:", len(results_df))

# ------------------------------------------------------------
# Build resume keys
# ------------------------------------------------------------
completed_keys = set(
    results_df["ID"].astype(str)
    + "_"
    + results_df["Language"].astype(str)
)

# ------------------------------------------------------------
# Classification function
# ------------------------------------------------------------
def classify_statement(statement, language):

    if language == "English":

        prompt = f"""Decide whether this statement is TRUE or FALSE.

Statement:
{statement}

Reply with exactly one word:
TRUE
or
FALSE"""

    else:

        prompt = f"""यो कथन TRUE हो कि FALSE हो?

कथन:
{statement}

ठ्याक्कै एउटा शब्द मात्र लेख्नुहोस्:
TRUE
वा
FALSE"""

    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "num_ctx": 512,
            "temperature": 0
        },
        keep_alive=0
    )

    answer = response.message.content.strip().upper()

    # Normalize output
    if "TRUE" in answer and "FALSE" not in answer:
        return "TRUE"

    if "FALSE" in answer and "TRUE" not in answer:
        return "FALSE"

    return answer

# ------------------------------------------------------------
# Process all 11,999 statements
# ------------------------------------------------------------
for index, row in poly_df.iterrows():

    row_id = str(row["ID"])
    gold_label = str(row["Label"]).strip().upper()

    # ========================================================
    # English
    # ========================================================
    english_key = row_id + "_English"

    if english_key not in completed_keys:

        try:

            statement = str(row["en"])

            prediction = classify_statement(
                statement,
                "English"
            )

            new_row = pd.DataFrame([{
                "ID": row_id,
                "Language": "English",
                "Statement": statement,
                "Gold Label": gold_label,
                "Gemma Prediction": prediction
            }])

            results_df = pd.concat(
                [results_df, new_row],
                ignore_index=True
            )

            completed_keys.add(english_key)

            results_df.to_csv(
                OUTPUT_FILE,
                index=False,
                encoding="utf-8-sig"
            )

            print(
                f"[{index+1}/{len(poly_df)}] "
                f"ID {row_id} English ✓"
            )

        except Exception as e:

            print(
                f"ERROR ID {row_id} English: {e}"
            )

            time.sleep(2)

    # ========================================================
    # Nepali
    # ========================================================
    nepali_key = row_id + "_Nepali"

    if nepali_key not in completed_keys:

        try:

            statement = str(row["Nepali"])

            prediction = classify_statement(
                statement,
                "Nepali"
            )

            new_row = pd.DataFrame([{
                "ID": row_id,
                "Language": "Nepali",
                "Statement": statement,
                "Gold Label": gold_label,
                "Gemma Prediction": prediction
            }])

            results_df = pd.concat(
                [results_df, new_row],
                ignore_index=True
            )

            completed_keys.add(nepali_key)

            results_df.to_csv(
                OUTPUT_FILE,
                index=False,
                encoding="utf-8-sig"
            )

            print(
                f"[{index+1}/{len(poly_df)}] "
                f"ID {row_id} Nepali ✓"
            )

        except Exception as e:

            print(
                f"ERROR ID {row_id} Nepali: {e}"
            )

            time.sleep(2)

print("\n" + "=" * 70)
print("POLY-FEVER COMPLETE")
print("=" * 70)
print("Total predictions:", len(results_df))
print("Expected:", len(poly_df) * 2)
print("Saved to:", OUTPUT_FILE)

Poly-FEVER rows: 11999
Existing predictions: 0
[1/11999] ID 1 English ✓
[1/11999] ID 1 Nepali ✓
[2/11999] ID 2 English ✓
[2/11999] ID 2 Nepali ✓
[3/11999] ID 3 English ✓
[3/11999] ID 3 Nepali ✓
[4/11999] ID 4 English ✓
[4/11999] ID 4 Nepali ✓
[5/11999] ID 5 English ✓
[5/11999] ID 5 Nepali ✓
[6/11999] ID 6 English ✓
[6/11999] ID 6 Nepali ✓
[7/11999] ID 7 English ✓
[7/11999] ID 7 Nepali ✓
[8/11999] ID 8 English ✓
[8/11999] ID 8 Nepali ✓
[9/11999] ID 9 English ✓
[9/11999] ID 9 Nepali ✓
[10/11999] ID 10 English ✓
[10/11999] ID 10 Nepali ✓
[11/11999] ID 11 English ✓
[11/11999] ID 11 Nepali ✓
[12/11999] ID 12 English ✓
[12/11999] ID 12 Nepali ✓
[13/11999] ID 13 English ✓
[13/11999] ID 13 Nepali ✓
[14/11999] ID 14 English ✓
[14/11999] ID 14 Nepali ✓
[15/11999] ID 15 English ✓
[15/11999] ID 15 Nepali ✓
[16/11999] ID 16 English ✓
[16/11999] ID 16 Nepali ✓
[17/11999] ID 17 English ✓
[17/11999] ID 17 Nepali ✓
[18/11999] ID 18 English ✓
[18/11999] ID 18 Nepali ✓
[19/11999] ID 19 English ✓
[19/1199

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import ollama
import os
import json
import re

EXCEL_FILE = r"E:\Desktop\Research_Hallucination\Hexagon-Dataset.xlsx"
OUTPUT_FILE = r"E:\Desktop\Research_Hallucination\polyfever_gemma_results.csv"

MODEL = "gemma3:4b"
BATCH_SIZE = 20

# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------
poly_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Poly-FEVER"
)

print("Poly-FEVER rows:", len(poly_df))

# ------------------------------------------------------------
# Load existing results
# ------------------------------------------------------------
columns = [
    "ID",
    "Language",
    "Statement",
    "Gold Label",
    "Gemma Prediction"
]

if os.path.exists(OUTPUT_FILE):
    results_df = pd.read_csv(OUTPUT_FILE)

    if all(c in results_df.columns for c in columns):
        results_df = results_df[columns].copy()
    else:
        results_df = pd.DataFrame(columns=columns)
else:
    results_df = pd.DataFrame(columns=columns)

print("Existing predictions:", len(results_df))

completed = set(
    results_df["ID"].astype(str)
    + "_"
    + results_df["Language"].astype(str)
)

# ------------------------------------------------------------
# Batch classifier
# ------------------------------------------------------------
def classify_batch(items, language):

    if language == "English":
        intro = (
            "Classify each statement as TRUE or FALSE.\n"
            "Return exactly one label per line, in the same order.\n"
            "Do not explain anything.\n\n"
        )
    else:
        intro = (
            "प्रत्येक कथन TRUE वा FALSE भनेर वर्गीकरण गर्नुहोस्।\n"
            "उही क्रमअनुसार प्रत्येक लाइनमा एउटा मात्र label दिनुहोस्।\n"
            "कुनै व्याख्या नदिनुहोस्।\n\n"
        )

    prompt = intro

    for i, item in enumerate(items, start=1):
        prompt += f"{i}. {item}\n"

    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "num_ctx": 2048,
            "temperature": 0
        },
        keep_alive="10m"
    )

    text = response.message.content.strip()

    # Extract TRUE/FALSE labels
    labels = re.findall(
        r'\b(TRUE|FALSE)\b',
        text.upper()
    )

    if len(labels) < len(items):
        return labels + ["UNKNOWN"] * (len(items) - len(labels))

    return labels[:len(items)]


# ------------------------------------------------------------
# Process one language at a time
# ------------------------------------------------------------
for language, column in [
    ("English", "en"),
    ("Nepali", "Nepali")
]:

    print("\n" + "=" * 70)
    print("LANGUAGE:", language)
    print("=" * 70)

    pending = []

    for _, row in poly_df.iterrows():

        key = str(row["ID"]) + "_" + language

        if key not in completed:

            pending.append({
                "ID": str(row["ID"]),
                "Statement": str(row[column]),
                "Gold Label": str(row["Label"]).strip().upper()
            })

    print("Pending:", len(pending))

    for start in range(0, len(pending), BATCH_SIZE):

        batch = pending[start:start + BATCH_SIZE]

        statements = [
            item["Statement"]
            for item in batch
        ]

        try:
            predictions = classify_batch(
                statements,
                language
            )

            new_rows = []

            for item, prediction in zip(batch, predictions):

                new_rows.append({
                    "ID": item["ID"],
                    "Language": language,
                    "Statement": item["Statement"],
                    "Gold Label": item["Gold Label"],
                    "Gemma Prediction": prediction
                })

            new_df = pd.DataFrame(new_rows)

            results_df = pd.concat(
                [results_df, new_df],
                ignore_index=True
            )

            # Save after every batch
            results_df.to_csv(
                OUTPUT_FILE,
                index=False,
                encoding="utf-8-sig"
            )

            for item in batch:
                completed.add(
                    item["ID"] + "_" + language
                )

            print(
                f"{language}: "
                f"{min(start + BATCH_SIZE, len(pending))}/{len(pending)} "
                f"completed"
            )

        except Exception as e:

            print(
                f"Batch error at {start}: {e}"
            )

            # Wait briefly before continuing
            import time
            time.sleep(3)

print("\n" + "=" * 70)
print("POLY-FEVER BATCH RUN COMPLETE")
print("=" * 70)

print("Total predictions:", len(results_df))
print("Expected:", len(poly_df) * 2)
print("Saved:", OUTPUT_FILE)

Poly-FEVER rows: 11999
Existing predictions: 18398

LANGUAGE: English
Pending: 0

LANGUAGE: Nepali
Pending: 5600
Nepali: 20/5600 completed
Nepali: 40/5600 completed
Nepali: 60/5600 completed
Nepali: 80/5600 completed
Nepali: 100/5600 completed
Nepali: 120/5600 completed
Nepali: 140/5600 completed
Nepali: 160/5600 completed
Nepali: 180/5600 completed
Nepali: 200/5600 completed
Nepali: 220/5600 completed
Nepali: 240/5600 completed
Nepali: 260/5600 completed
Nepali: 280/5600 completed
Nepali: 300/5600 completed
Nepali: 320/5600 completed
Nepali: 340/5600 completed
Nepali: 360/5600 completed
Nepali: 380/5600 completed
Nepali: 400/5600 completed
Nepali: 420/5600 completed
Nepali: 440/5600 completed
Nepali: 460/5600 completed
Nepali: 480/5600 completed
Nepali: 500/5600 completed
Nepali: 520/5600 completed
Nepali: 540/5600 completed
Nepali: 560/5600 completed
Nepali: 580/5600 completed
Nepali: 600/5600 completed
Nepali: 620/5600 completed
Nepali: 640/5600 completed
Nepali: 660/5600 completed


In [ ]:
import pandas as pd
import os

POLY_RESULTS = r"E:\Desktop\Research_Hallucination\polyfever_gemma_results.csv"

df_poly = pd.read_csv(POLY_RESULTS)

print("Total predictions:", len(df_poly))
print("\nBy language:")
print(df_poly["Language"].value_counts())

# Normalize labels
df_poly["Gold Label"] = (
    df_poly["Gold Label"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_poly["Gemma Prediction"] = (
    df_poly["Gemma Prediction"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Correctness
df_poly["Correct"] = (
    df_poly["Gold Label"] ==
    df_poly["Gemma Prediction"]
)

# Hallucination:
# FALSE ground truth predicted as TRUE
df_poly["Hallucination"] = (
    (df_poly["Gold Label"] == "FALSE") &
    (df_poly["Gemma Prediction"] == "TRUE")
)

print("\nOverall accuracy:")
print(df_poly["Correct"].mean() * 100)

print("\nOverall hallucination rate:")
false_cases = df_poly[df_poly["Gold Label"] == "FALSE"]

if len(false_cases) > 0:
    rate = false_cases["Hallucination"].mean() * 100
    print(rate)
else:
    print("No FALSE cases found.")

print("\nBy language:")

for language in ["English", "Nepali"]:

    subset = df_poly[
        df_poly["Language"] == language
    ]

    accuracy = subset["Correct"].mean() * 100

    false_subset = subset[
        subset["Gold Label"] == "FALSE"
    ]

    hallucination_rate = (
        false_subset["Hallucination"].mean() * 100
        if len(false_subset)
        else 0
    )

    print(
        f"{language}: "
        f"{len(subset)} responses | "
        f"Accuracy: {accuracy:.2f}% | "
        f"Hallucination: {hallucination_rate:.2f}%"
    )

Total predictions: 23998

By language:
Language
English    11999
Nepali     11999
Name: count, dtype: int64

Overall accuracy:
52.8002333527794

Overall hallucination rate:
51.52772538664655

By language:
English: 11999 responses | Accuracy: 57.14% | Hallucination: 66.45%
Nepali: 11999 responses | Accuracy: 48.46% | Hallucination: 36.61%


In [ ]:
import pandas as pd
import os

FOLDER = r"E:\Desktop\Research_Hallucination\gemma_results"

files = [
    "Constitution_Law_gemma_results.csv",
    "Culture_Religion_Literature_gemma_results.csv",
    "Geography and tourism_gemma_results.csv",
    "Education health_gemma_results.csv",
    "Economy _ Technology _ Miscella_gemma_results.csv"
]

frames = []

for filename in files:
    path = os.path.join(FOLDER, filename)
    temp = pd.read_csv(path, encoding="utf-8-sig")
    temp["Dataset"] = filename.replace("_gemma_results.csv", "")
    frames.append(temp)

qa_results = pd.concat(frames, ignore_index=True)

print("Total Q&A responses:", len(qa_results))
print("\nBy language:")
print(qa_results["Language"].value_counts())

Total Q&A responses: 3706

By language:
Language
Nepali     1853
English    1853
Name: count, dtype: int64


In [ ]:
import ollama
import json
import re

EVALUATOR_PROMPT = """
You are evaluating an AI-generated answer against a provided gold answer.

For each item, determine:

VERDICT:
- Correct
- Error

ERROR TYPE:
- None
- Hallucinated / unsupported elaboration
- Incorrect factual / unsupported claim
- No answer / unsupported-context refusal
- Question repetition / non-answer

Rules:
1. Do not require exact wording.
2. Judge whether the core answer is correct.
3. Extra information is NOT automatically an error.
4. Use "Hallucinated / unsupported elaboration" when the core answer is correct but the response adds unsupported factual claims or unnecessary invented details.
5. Use "Incorrect factual / unsupported claim" when the answer contains a materially wrong factual claim or gives the wrong answer.
6. Use "No answer / unsupported-context refusal" when the model refuses, says it cannot answer, or provides no usable answer.
7. Use "Question repetition / non-answer" when the model mainly repeats the question or otherwise fails to answer.
8. If the response is correct and sufficiently supported, use Correct and None.

Return EXACTLY one line per item using this format:

ITEM 1 | Verdict: Correct | Error Type: None
"""

sample = qa_results.head(5)

prompt = EVALUATOR_PROMPT + "\n\n"

for i, (_, row) in enumerate(sample.iterrows(), start=1):
    prompt += f"""
ITEM {i}
Language: {row["Language"]}
Question: {row["Question"]}
Gold Answer: {row["Gold Answer"]}
AI Answer: {row["Gemma Answer"]}
"""

response = ollama.chat(
    model="gemma3:4b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    options={
        "num_ctx": 4096,
        "temperature": 0
    },
    keep_alive="10m"
)

print(response.message.content)

ResponseError: model requires more system memory (4.0 GiB) than is available (3.7 GiB) (status code: 500)

In [ ]:
import psutil

ram = psutil.virtual_memory()

print("Total RAM:", round(ram.total / 1024**3, 2), "GB")
print("Available RAM:", round(ram.available / 1024**3, 2), "GB")
print("Used RAM:", round(ram.used / 1024**3, 2), "GB")

Total RAM: 15.71 GB
Available RAM: 1.29 GB
Used RAM: 14.42 GB


In [ ]:
import psutil

ram = psutil.virtual_memory()

print("Available RAM:", round(ram.available / 1024**3, 2), "GB")


Available RAM: 1.52 GB


In [ ]:
import pandas as pd
import glob
import os

FOLDER = r"E:\Desktop\Research_Hallucination\gemma_results"

files = glob.glob(os.path.join(FOLDER, "*.csv"))

qa_frames = []

for file in files:
    df = pd.read_csv(file, encoding="utf-8-sig")

    # Only use the Q&A result files
    required = {
        "Question ID",
        "Language",
        "Question",
        "Gold Answer",
        "Gemma Answer"
    }

    if required.issubset(df.columns):
        df["Dataset"] = os.path.basename(file)
        qa_frames.append(df)

qa_results = pd.concat(
    qa_frames,
    ignore_index=True
)

# Add evaluation columns
qa_results["Verdict"] = ""
qa_results["Error Type"] = ""

print("Total Q&A responses:", len(qa_results))
print("\nBy dataset:")
print(qa_results["Dataset"].value_counts())

print("\nBy language:")
print(qa_results["Language"].value_counts())

Total Q&A responses: 3706

By dataset:
Dataset
Constitution_Law_gemma_results.csv                   1810
Culture_Religion_Literature_gemma_results.csv         848
Geography and tourism_gemma_results.csv               466
Education health_gemma_results.csv                    316
Economy _ Technology _ Miscella_gemma_results.csv     266
Name: count, dtype: int64

By language:
Language
Nepali     1853
English    1853
Name: count, dtype: int64


In [ ]:
import pandas as pd
import re

# Work on a copy so raw results stay untouched
eval_df = qa_results.copy()

eval_df["Verdict"] = "Review Needed"
eval_df["Error Type"] = "Review Needed"

# ------------------------------------------------------------
# Basic text normalization
# ------------------------------------------------------------
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

# ------------------------------------------------------------
# Refusal / no-answer detection
# ------------------------------------------------------------
REFUSAL_PHRASES = [
    "i don't know",
    "i do not know",
    "i cannot answer",
    "i can't answer",
    "i cannot provide",
    "i'm unable to answer",
    "i am unable to answer",
    "i don't have enough information",
    "cannot determine",
    "unable to determine",
    "मलाई थाहा छैन",
    "म जानकारी दिन सक्दिन",
    "उत्तर दिन सक्दिन"
]

def is_refusal(text):
    text = normalize_text(text)
    return any(p in text for p in REFUSAL_PHRASES)

# ------------------------------------------------------------
# Question repetition / non-answer detection
# ------------------------------------------------------------
def is_question_repetition(question, answer):
    q = normalize_text(question)
    a = normalize_text(answer)

    if not q or not a:
        return False

    # Very high overlap with the question and very little extra content
    q_words = set(re.findall(r"\w+", q))
    a_words = set(re.findall(r"\w+", a))

    if len(a_words) <= 3:
        return False

    overlap = len(q_words & a_words) / max(len(q_words), 1)

    return overlap > 0.80 and len(a_words) <= len(q_words) * 1.3

# ------------------------------------------------------------
# Gold-answer presence check
# ------------------------------------------------------------
def gold_terms(text):
    text = normalize_text(text)

    # Keep useful alphanumeric tokens
    return set(re.findall(r"[a-z0-9\u0900-\u097f]+", text))

def answer_contains_gold(gold, answer):
    gold_tokens = gold_terms(gold)
    answer_tokens = gold_terms(answer)

    if not gold_tokens:
        return False

    # Exact phrase
    g = normalize_text(gold)
    a = normalize_text(answer)

    if g in a:
        return True

    # If most gold tokens appear in the answer
    overlap = len(gold_tokens & answer_tokens) / len(gold_tokens)

    return overlap >= 0.75

# ------------------------------------------------------------
# First-pass classification
# ------------------------------------------------------------
for i, row in eval_df.iterrows():

    question = row["Question"]
    gold = row["Gold Answer"]
    answer = row["Gemma Answer"]

    # Empty answer
    if not str(answer).strip() or str(answer).strip().lower() == "nan":
        eval_df.at[i, "Verdict"] = "Error"
        eval_df.at[i, "Error Type"] = "No answer / unsupported-context refusal"
        continue

    # Refusal
    if is_refusal(answer):
        eval_df.at[i, "Verdict"] = "Error"
        eval_df.at[i, "Error Type"] = "No answer / unsupported-context refusal"
        continue

    # Question repetition
    if is_question_repetition(question, answer):
        eval_df.at[i, "Verdict"] = "Error"
        eval_df.at[i, "Error Type"] = "Question repetition / non-answer"
        continue

    # Gold answer explicitly present
    if answer_contains_gold(gold, answer):
        eval_df.at[i, "Verdict"] = "Correct"
        eval_df.at[i, "Error Type"] = "None"
        continue

print("First-pass evaluation complete.")

print("\nVerdict counts:")
print(eval_df["Verdict"].value_counts())

print("\nError-type counts:")
print(eval_df["Error Type"].value_counts())

First-pass evaluation complete.

Verdict counts:
Verdict
Review Needed    3307
Correct           315
Error              84
Name: count, dtype: int64

Error-type counts:
Error Type
Review Needed                              3307
None                                        315
Question repetition / non-answer             82
No answer / unsupported-context refusal       2
Name: count, dtype: int64


In [ ]:
import os
import requests

# Put your key in this variable temporarily.
# DO NOT send the key to me.
OPENROUTER_API_KEY = "PASTE_YOUR_KEY_HERE"

url = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}

data = {
    "model": "openai/gpt-oss-20b:free",
    "messages": [
        {
            "role": "user",
            "content": "Reply with exactly: TEST SUCCESS"
        }
    ],
    "temperature": 0,
    "max_tokens": 10
}

response = requests.post(
    url,
    headers=headers,
    json=data,
    timeout=60
)

print("Status:", response.status_code)
print(response.text[:1000])

Status: 401
{"error":{"message":"Missing Authentication header","code":401}}


In [ ]:
import requests

API_KEY = "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"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

payload = {
    "model": "openai/gpt-oss-20b:free",
    "messages": [
        {
            "role": "user",
            "content": "Reply with exactly: TEST SUCCESS"
        }
    ],
    "temperature": 0,
    "max_tokens": 10
}

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers=headers,
    json=payload,
    timeout=60
)

print("Status:", response.status_code)

if response.status_code == 200:
    print(response.json()["choices"][0]["message"]["content"])
else:
    print(response.text)

Status: 404
{"error":{"message":"This model is unavailable for free. The paid version is available now - use this slug instead: openai/gpt-oss-20b","code":404},"user_id":"user_3DKrB08En3IgTzGQn4gfq34ZijG"}


In [ ]:
import requests

API_KEY = "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"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

payload = {
    "model": "openrouter/free",
    "messages": [
        {
            "role": "user",
            "content": "Reply with exactly: TEST SUCCESS"
        }
    ],
    "temperature": 0,
    "max_tokens": 10
}

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers=headers,
    json=payload,
    timeout=60
)

print("Status:", response.status_code)

if response.status_code == 200:
    data = response.json()
    print("Model used:", data.get("model"))
    print("Answer:", data["choices"][0]["message"]["content"])
else:
    print(response.text)

Status: 200
Model used: nvidia/nemotron-3-ultra-550b-a55b:free
Answer: The user wants me to reply with exactly "TEST


In [ ]:
import requests
import json

API_KEY = "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"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

test_prompt = """
Evaluate these two AI answers.

Allowed Verdict:
- Correct
- Error

Allowed Error Type:
- None
- Hallucinated / unsupported elaboration
- Incorrect factual / unsupported claim
- No answer / unsupported-context refusal
- Question repetition / non-answer

Rules:
- Judge the core answer, not exact wording.
- If the core answer is correct and extra information is harmless/supported, use Correct.
- If the core answer is correct but the answer contains unsupported factual additions, use Hallucinated / unsupported elaboration.
- If the answer contains a materially false factual claim or gives the wrong answer, use Incorrect factual / unsupported claim.
- If the model refuses or gives no usable answer, use No answer / unsupported-context refusal.
- If it repeats the question instead of answering, use Question repetition / non-answer.

Return ONLY valid JSON in this exact structure:

[
  {
    "item": 1,
    "verdict": "Correct",
    "error_type": "None"
  }
]

Do not include markdown or explanations.

ITEM 1
Question: What is the official currency of Nepal?
Gold Answer: Nepalese Rupee
AI Answer: The official currency of Nepal is the Nepalese Rupee (NPR).

ITEM 2
Question: In which year was Nepal Bank Limited established?
Gold Answer: 1937 (1994 BS)
AI Answer: Nepal Bank Limited was established in 1963.
"""

payload = {
    "model": "openrouter/free",
    "messages": [
        {
            "role": "user",
            "content": test_prompt
        }
    ],
    "temperature": 0,
    "max_tokens": 200
}

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers=headers,
    json=payload,
    timeout=120
)

print("Status:", response.status_code)

if response.status_code == 200:
    data = response.json()
    print("Model used:", data.get("model"))
    print("\nRaw evaluator output:")
    print(data["choices"][0]["message"]["content"])
else:
    print(response.text)

Status: 200
Model used: nvidia/nemotron-3-super-120b-a12b:free

Raw evaluator output:
[
  {



In [ ]:
import pandas as pd
import requests
import json
import os
import re
import time

# ============================================================
# SETTINGS
# ============================================================

INPUT_FOLDER = r"E:\Desktop\Research_Hallucination\gemma_results"
OUTPUT_FILE = r"E:\Desktop\Research_Hallucination\Q&A_final_evaluation.csv"

BATCH_SIZE = 75
MODEL = "openrouter/free"

API_URL = "https://openrouter.ai/api/v1/chat/completions"

# ============================================================
# LOAD THE 5 RAW Q&A FILES
# ============================================================

files = [
    "Constitution_Law_gemma_results.csv",
    "Culture_Religion_Literature_gemma_results.csv",
    "Geography and tourism_gemma_results.csv",
    "Education health_gemma_results.csv",
    "Economy _ Technology _ Miscella_gemma_results.csv"
]

frames = []

for filename in files:
    path = os.path.join(INPUT_FOLDER, filename)

    temp = pd.read_csv(
        path,
        encoding="utf-8-sig"
    )

    temp["Dataset"] = filename.replace(
        "_gemma_results.csv",
        ""
    )

    frames.append(temp)

qa_results = pd.concat(
    frames,
    ignore_index=True
)

print("Total Q&A responses:", len(qa_results))

# ============================================================
# LOAD PREVIOUS EVALUATION IF IT EXISTS
# ============================================================

required_output_columns = [
    "Question ID",
    "Language",
    "Question",
    "Gold Answer",
    "Gemma Answer",
    "Dataset",
    "Verdict",
    "Error Type"
]

if os.path.exists(OUTPUT_FILE):

    evaluation_df = pd.read_csv(
        OUTPUT_FILE,
        encoding="utf-8-sig"
    )

    print(
        "Existing evaluated rows:",
        len(evaluation_df)
    )

else:

    evaluation_df = pd.DataFrame(
        columns=required_output_columns
    )

    print("Starting new evaluation.")

# ============================================================
# COMPLETED KEYS
# ============================================================

completed_keys = set()

if len(evaluation_df) > 0:

    completed_keys = set(
        evaluation_df["Dataset"].astype(str)
        + "|"
        + evaluation_df["Question ID"].astype(str)
        + "|"
        + evaluation_df["Language"].astype(str)
    )

# ============================================================
# EVALUATOR PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are an AI-answer evaluator for a hallucination research study.

Evaluate each AI answer against the supplied question and gold answer.

Allowed Verdict:
- Correct
- Error

Allowed Error Type:
- None
- Hallucinated / unsupported elaboration
- Incorrect factual / unsupported claim
- No answer / unsupported-context refusal
- Question repetition / non-answer

Rules:

1. Do not require exact wording.
2. Judge semantic correctness of the answer.
3. A correct paraphrase is Correct.
4. If the core answer is correct but the response adds an unsupported factual claim or invented detail, use:
   Hallucinated / unsupported elaboration
5. If the response gives the wrong answer or contains a materially incorrect factual claim, use:
   Incorrect factual / unsupported claim
6. If the model refuses, says it does not know, or provides no usable answer, use:
   No answer / unsupported-context refusal
7. If the model mainly repeats the question or fails to answer it, use:
   Question repetition / non-answer
8. Do not classify harmless formatting differences as errors.
9. Return exactly one JSON object per item.
10. Do not provide explanations.

Required JSON format:

[
  {
    "item": 1,
    "verdict": "Correct",
    "error_type": "None"
  }
]
"""

# ============================================================
# OPENROUTER CALL
# ============================================================

def evaluate_batch(batch_df):

    prompt = SYSTEM_PROMPT + "\n\n"

    for item_number, (_, row) in enumerate(
        batch_df.iterrows(),
        start=1
    ):

        prompt += f"""
ITEM {item_number}

Language:
{row["Language"]}

Question:
{row["Question"]}

Gold Answer:
{row["Gold Answer"]}

AI Answer:
{row["Gemma Answer"]}

"""

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0,
        "max_tokens": 5000
    }

    response = requests.post(
        API_URL,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json"
        },
        json=payload,
        timeout=300
    )

    response.raise_for_status()

    data = response.json()

    text = data["choices"][0]["message"]["content"]

    return text, data.get("model", MODEL)

# ============================================================
# PARSE JSON
# ============================================================

def parse_json_output(text):

    text = text.strip()

    # Remove markdown code fences if the model added them
    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    # Find JSON array
    start = text.find("[")
    end = text.rfind("]")

    if start == -1 or end == -1:
        raise ValueError(
            "No JSON array found in evaluator output."
        )

    json_text = text[start:end + 1]

    return json.loads(json_text)

# ============================================================
# PREPARE PENDING ROWS
# ============================================================

pending_rows = []

for idx, row in qa_results.iterrows():

    key = (
        str(row["Dataset"])
        + "|"
        + str(row["Question ID"])
        + "|"
        + str(row["Language"])
    )

    if key not in completed_keys:
        pending_rows.append(idx)

print("Pending evaluations:", len(pending_rows))

# ============================================================
# BATCH EVALUATION
# ============================================================

for start in range(
    0,
    len(pending_rows),
    BATCH_SIZE
):

    batch_indices = pending_rows[
        start:start + BATCH_SIZE
    ]

    batch_df = qa_results.loc[
        batch_indices
    ].copy()

    print(
        f"\nEvaluating "
        f"{start + 1}-"
        f"{start + len(batch_df)} "
        f"of {len(pending_rows)}"
    )

    try:

        raw_output, model_used = evaluate_batch(
            batch_df
        )

        parsed = parse_json_output(
            raw_output
        )

        if len(parsed) != len(batch_df):

            raise ValueError(
                f"Expected {len(batch_df)} evaluations "
                f"but received {len(parsed)}."
            )

        new_rows = []

        for item_number, (
            (_, source_row),
            result
        ) in enumerate(
            zip(batch_df.iterrows(), parsed),
            start=1
        ):

            verdict = str(
                result.get("verdict", "")
            ).strip()

            error_type = str(
                result.get("error_type", "")
            ).strip()

            # Validate evaluator output
            if verdict not in [
                "Correct",
                "Error"
            ]:

                raise ValueError(
                    f"Invalid verdict: {verdict}"
                )

            allowed_errors = [
                "None",
                "Hallucinated / unsupported elaboration",
                "Incorrect factual / unsupported claim",
                "No answer / unsupported-context refusal",
                "Question repetition / non-answer"
            ]

            if error_type not in allowed_errors:

                raise ValueError(
                    f"Invalid error type: {error_type}"
                )

            new_rows.append({
                "Question ID":
                    source_row["Question ID"],
                "Language":
                    source_row["Language"],
                "Question":
                    source_row["Question"],
                "Gold Answer":
                    source_row["Gold Answer"],
                "Gemma Answer":
                    source_row["Gemma Answer"],
                "Dataset":
                    source_row["Dataset"],
                "Verdict":
                    verdict,
                "Error Type":
                    error_type,
                "Evaluator Model":
                    model_used
            })

        new_df = pd.DataFrame(
            new_rows
        )

        evaluation_df = pd.concat(
            [
                evaluation_df,
                new_df
            ],
            ignore_index=True
        )

        # SAVE AFTER EVERY BATCH
        evaluation_df.to_csv(
            OUTPUT_FILE,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            "Batch saved successfully."
        )

    except Exception as e:

        print(
            "\nBATCH ERROR:",
            type(e).__name__,
            str(e)
        )

        print(
            "This batch was NOT saved."
        )

        print(
            "You can rerun the cell and completed "
            "batches will be skipped."
        )

        break

    # Small delay to avoid hammering the API
    time.sleep(1)

# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("Q&A EVALUATION STATUS")
print("=" * 70)

print(
    "Evaluated rows:",
    len(evaluation_df)
)

print(
    "Target rows:",
    len(qa_results)
)

print(
    "Remaining:",
    len(qa_results) - len(evaluation_df)
)

print(
    "Saved to:",
    OUTPUT_FILE
)

if len(evaluation_df) == len(qa_results):

    print("\n✅ ALL Q&A RESPONSES EVALUATED!")

    print("\nVerdict:")
    print(
        evaluation_df["Verdict"]
        .value_counts()
    )

    print("\nError Type:")
    print(
        evaluation_df["Error Type"]
        .value_counts()
    )

Total Q&A responses: 3706
Starting new evaluation.
Pending evaluations: 3706

Evaluating 1-75 of 3706

BATCH ERROR: ValueError Invalid verdict: Incorrect factual / unsupported claim
This batch was NOT saved.
You can rerun the cell and completed batches will be skipped.

Q&A EVALUATION STATUS
Evaluated rows: 0
Target rows: 3706
Remaining: 3706
Saved to: E:\Desktop\Research_Hallucination\Q&A_final_evaluation.csv


In [ ]:
import pandas as pd
import requests
import json
import os
import re
import time

# ============================================================
# SETTINGS
# ============================================================

INPUT_FOLDER = r"E:\Desktop\Research_Hallucination\gemma_results"
OUTPUT_FILE = r"E:\Desktop\Research_Hallucination\Q&A_final_evaluation.csv"

BATCH_SIZE = 20
MODEL = "openrouter/free"
API_URL = "https://openrouter.ai/api/v1/chat/completions"

# ============================================================
# LOAD RAW Q&A RESULTS
# ============================================================

files = [
    "Constitution_Law_gemma_results.csv",
    "Culture_Religion_Literature_gemma_results.csv",
    "Geography and tourism_gemma_results.csv",
    "Education health_gemma_results.csv",
    "Economy _ Technology _ Miscella_gemma_results.csv"
]

frames = []

for filename in files:
    path = os.path.join(INPUT_FOLDER, filename)

    temp = pd.read_csv(
        path,
        encoding="utf-8-sig"
    )

    temp["Dataset"] = filename.replace(
        "_gemma_results.csv",
        ""
    )

    frames.append(temp)

qa_results = pd.concat(
    frames,
    ignore_index=True
)

print("Total Q&A responses:", len(qa_results))

# ============================================================
# LOAD PREVIOUS RESULTS
# ============================================================

output_columns = [
    "Question ID",
    "Language",
    "Question",
    "Gold Answer",
    "Gemma Answer",
    "Dataset",
    "Verdict",
    "Error Type",
    "Evaluator Model"
]

if os.path.exists(OUTPUT_FILE):

    try:
        evaluation_df = pd.read_csv(
            OUTPUT_FILE,
            encoding="utf-8-sig"
        )

        if "Question ID" in evaluation_df.columns:

            print(
                "Existing evaluated rows:",
                len(evaluation_df)
            )

        else:
            evaluation_df = pd.DataFrame(
                columns=output_columns
            )

    except Exception:
        evaluation_df = pd.DataFrame(
            columns=output_columns
        )

else:

    evaluation_df = pd.DataFrame(
        columns=output_columns
    )

# ============================================================
# COMPLETED KEYS
# ============================================================

completed_keys = set()

if len(evaluation_df) > 0:

    completed_keys = set(
        evaluation_df["Dataset"].astype(str)
        + "|"
        + evaluation_df["Question ID"].astype(str)
        + "|"
        + evaluation_df["Language"].astype(str)
    )

# ============================================================
# STRICT EVALUATOR
# ============================================================

SYSTEM_PROMPT = """
You are evaluating AI-generated answers for a research study.

For EACH item, choose exactly ONE CODE.

C = Correct answer

E1 = Hallucinated / unsupported elaboration
The core answer is correct, but the response adds unsupported or invented factual information.

E2 = Incorrect factual / unsupported claim
The core answer is wrong OR the response contains a materially false factual claim.

E3 = No answer / unsupported-context refusal
The model refuses, says it does not know, or provides no usable answer.

E4 = Question repetition / non-answer
The model mainly repeats the question or otherwise fails to answer.

Rules:
- Do not require exact wording.
- Accept valid paraphrases.
- Do not mark harmless extra wording as an error.
- Use E1 when the core answer is correct but unsupported factual additions are present.
- Use E2 when there is a materially incorrect factual claim or wrong answer.
- Return ONLY a JSON array.
- Return exactly one object for every item.
- Never put an error category inside the code field.

Required format:

[
  {"item": 1, "code": "C"},
  {"item": 2, "code": "E2"}
]
"""

# ============================================================
# OPENROUTER
# ============================================================

def call_evaluator(batch):

    prompt = SYSTEM_PROMPT + "\n\n"

    for i, (_, row) in enumerate(
        batch.iterrows(),
        start=1
    ):

        prompt += f"""
ITEM {i}
Language: {row["Language"]}
Question: {row["Question"]}
Gold Answer: {row["Gold Answer"]}
AI Answer: {row["Gemma Answer"]}
"""

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0,
        "max_tokens": 3000
    }

    r = requests.post(
        API_URL,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json"
        },
        json=payload,
        timeout=300
    )

    r.raise_for_status()

    data = r.json()

    return (
        data["choices"][0]["message"]["content"],
        data.get("model", MODEL)
    )

# ============================================================
# PARSE
# ============================================================

def parse_codes(text, expected):

    text = text.strip()

    # Normalize
    upper = text.upper()

    # --------------------------------------------------------
    # 1. Try JSON first
    # --------------------------------------------------------
    start = upper.find("[")
    end = upper.rfind("]")

    if start != -1 and end != -1:
        try:
            data = json.loads(text[start:end + 1])

            if len(data) == expected:
                codes = []

                for item in data:
                    code = str(
                        item.get("code", "")
                    ).strip().upper()

                    if code in {"C", "E1", "E2", "E3", "E4"}:
                        codes.append(code)

                if len(codes) == expected:
                    return codes

        except Exception:
            pass

    # --------------------------------------------------------
    # 2. Try explicit CODE labels
    # --------------------------------------------------------
    found_codes = re.findall(
        r'\b(E[1-4]|C)\b',
        upper
    )

    if len(found_codes) >= expected:
        return found_codes[:expected]

    # --------------------------------------------------------
    # 3. Try "Correct" / "Error" + error type text
    # --------------------------------------------------------
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    codes = []

    for line in lines:

        line_upper = line.upper()

        # Most specific errors first
        if (
            "HALLUCINATED / UNSUPPORTED ELABORATION"
            in line_upper
        ):
            codes.append("E1")

        elif (
            "INCORRECT FACTUAL / UNSUPPORTED CLAIM"
            in line_upper
        ):
            codes.append("E2")

        elif (
            "NO ANSWER / UNSUPPORTED-CONTEXT REFUSAL"
            in line_upper
            or "NO ANSWER / UNSUPPORTED CONTEXT REFUSAL"
            in line_upper
        ):
            codes.append("E3")

        elif (
            "QUESTION REPETITION / NON-ANSWER"
            in line_upper
        ):
            codes.append("E4")

        elif re.search(r'\bCORRECT\b', line_upper):
            codes.append("C")

    if len(codes) >= expected:
        return codes[:expected]

    # --------------------------------------------------------
    # Nothing matched
    # --------------------------------------------------------
    raise ValueError(
        "Could not extract enough evaluation codes.\n\n"
        "Raw evaluator output:\n"
        + text[:5000]
    )

# ============================================================
# CODE → RESEARCH LABEL
# ============================================================

label_map = {
    "C": (
        "Correct",
        "None"
    ),

    "E1": (
        "Error",
        "Hallucinated / unsupported elaboration"
    ),

    "E2": (
        "Error",
        "Incorrect factual / unsupported claim"
    ),

    "E3": (
        "Error",
        "No answer / unsupported-context refusal"
    ),

    "E4": (
        "Error",
        "Question repetition / non-answer"
    )
}

# ============================================================
# FIND PENDING
# ============================================================

pending_indices = []

for idx, row in qa_results.iterrows():

    key = (
        str(row["Dataset"])
        + "|"
        + str(row["Question ID"])
        + "|"
        + str(row["Language"])
    )

    if key not in completed_keys:
        pending_indices.append(idx)

print(
    "Pending evaluations:",
    len(pending_indices)
)

# ============================================================
# RUN BATCHES
# ============================================================

for start in range(
    0,
    len(pending_indices),
    BATCH_SIZE
):

    batch_indices = pending_indices[
        start:start + BATCH_SIZE
    ]

    batch = qa_results.loc[
        batch_indices
    ].copy()

    print(
        f"\nEvaluating "
        f"{start + 1}-"
        f"{start + len(batch)} "
        f"of {len(pending_indices)}"
    )

    try:

        raw_output, evaluator_model = call_evaluator(
            batch
        )

        codes = parse_codes(
            raw_output,
            len(batch)
        )

        new_rows = []

        for (_, row), code in zip(
            batch.iterrows(),
            codes
        ):

            verdict, error_type = label_map[code]

            new_rows.append({
                "Question ID": row["Question ID"],
                "Language": row["Language"],
                "Question": row["Question"],
                "Gold Answer": row["Gold Answer"],
                "Gemma Answer": row["Gemma Answer"],
                "Dataset": row["Dataset"],
                "Verdict": verdict,
                "Error Type": error_type,
                "Evaluator Model": evaluator_model
            })

        evaluation_df = pd.concat(
            [
                evaluation_df,
                pd.DataFrame(new_rows)
            ],
            ignore_index=True
        )

        evaluation_df.to_csv(
            OUTPUT_FILE,
            index=False,
            encoding="utf-8-sig"
        )

        print("✅ Batch saved.")

    except Exception as e:

        print(
            "\n❌ Batch failed:",
            type(e).__name__,
            str(e)
        )

        print(
            "No rows from this batch were saved."
        )

        break

print("\n" + "=" * 70)
print("EVALUATION STATUS")
print("=" * 70)

print(
    "Evaluated:",
    len(evaluation_df)
)

print(
    "Target:",
    len(qa_results)
)

print(
    "Remaining:",
    len(qa_results) - len(evaluation_df)
)

print(
    "Output:",
    OUTPUT_FILE
)

Total Q&A responses: 3706
Pending evaluations: 3706

Evaluating 1-20 of 3706

❌ Batch failed: AttributeError 'NoneType' object has no attribute 'strip'
No rows from this batch were saved.

EVALUATION STATUS
Evaluated: 0
Target: 3706
Remaining: 3706
Output: E:\Desktop\Research_Hallucination\Q&A_final_evaluation.csv


In [ ]:
import requests
import json

test_prompt = """
Evaluate these two AI answers.

Use exactly one code per item:

C  = Correct
E1 = Hallucinated / unsupported elaboration
E2 = Incorrect factual / unsupported claim
E3 = No answer / unsupported-context refusal
E4 = Question repetition / non-answer

Rules:
- Accept valid paraphrases.
- C means the core answer is correct.
- E1 means the core answer is correct but unsupported factual additions are present.
- E2 means the answer is materially wrong or contains a materially false factual claim.
- E3 means refusal/no usable answer.
- E4 means repetition/non-answer.

Return one JSON object containing an array named "results".
"""

test_prompt += """

ITEM 1
Question: What is the official currency of Nepal?
Gold Answer: Nepalese Rupee
AI Answer: The official currency of Nepal is the Nepalese Rupee (NPR).

ITEM 2
Question: In which year was Nepal Bank Limited established?
Gold Answer: 1937 (1994 BS)
AI Answer: Nepal Bank Limited was established in 1963.
"""

payload = {
    "model": "openai/gpt-oss-20b",
    "messages": [
        {
            "role": "user",
            "content": test_prompt
        }
    ],
    "temperature": 0,
    "max_tokens": 300,
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "evaluation",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "results": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "item": {
                                    "type": "integer"
                                },
                                "code": {
                                    "type": "string",
                                    "enum": ["C", "E1", "E2", "E3", "E4"]
                                }
                            },
                            "required": ["item", "code"],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["results"],
                "additionalProperties": False
            }
        }
    },
    "reasoning": {
        "effort": "low",
        "exclude": True
    }
}

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    },
    json=payload,
    timeout=180
)

print("Status:", response.status_code)

if response.status_code == 200:
    data = response.json()

    print("Model:", data.get("model"))
    print("Message content:")
    print(data["choices"][0]["message"].get("content"))
else:
    print(response.text)

Status: 200
Model: openai/gpt-oss-20b
Message content:
{"results":[{"code":"C","item":1},{"code":"E2","item":2}]}


In [ ]:
import pandas as pd
import requests
import json
import os
import time

# ============================================================
# SETTINGS
# ============================================================

INPUT_FOLDER = r"E:\Desktop\Research_Hallucination\gemma_results"
OUTPUT_FILE = r"E:\Desktop\Research_Hallucination\Q&A_final_evaluation.csv"

API_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "openai/gpt-oss-20b"

BATCH_SIZE = 50

# ============================================================
# LOAD RAW Q&A RESULTS
# ============================================================

files = [
    "Constitution_Law_gemma_results.csv",
    "Culture_Religion_Literature_gemma_results.csv",
    "Geography and tourism_gemma_results.csv",
    "Education health_gemma_results.csv",
    "Economy _ Technology _ Miscella_gemma_results.csv"
]

frames = []

for filename in files:

    path = os.path.join(INPUT_FOLDER, filename)

    temp = pd.read_csv(
        path,
        encoding="utf-8-sig"
    )

    temp["Dataset"] = filename.replace(
        "_gemma_results.csv",
        ""
    )

    frames.append(temp)

qa_results = pd.concat(
    frames,
    ignore_index=True
)

print("Total Q&A responses:", len(qa_results))

# ============================================================
# LOAD PREVIOUS EVALUATION
# ============================================================

output_columns = [
    "Question ID",
    "Language",
    "Question",
    "Gold Answer",
    "Gemma Answer",
    "Dataset",
    "Verdict",
    "Error Type",
    "Evaluator Model"
]

if os.path.exists(OUTPUT_FILE):

    try:

        evaluation_df = pd.read_csv(
            OUTPUT_FILE,
            encoding="utf-8-sig"
        )

        if not all(
            c in evaluation_df.columns
            for c in output_columns
        ):
            evaluation_df = pd.DataFrame(
                columns=output_columns
            )

    except Exception:

        evaluation_df = pd.DataFrame(
            columns=output_columns
        )

else:

    evaluation_df = pd.DataFrame(
        columns=output_columns
    )

print(
    "Already evaluated:",
    len(evaluation_df)
)

# ============================================================
# BUILD RESUME KEYS
# ============================================================

completed_keys = set()

if len(evaluation_df) > 0:

    completed_keys = set(
        evaluation_df["Dataset"].astype(str)
        + "|"
        + evaluation_df["Question ID"].astype(str)
        + "|"
        + evaluation_df["Language"].astype(str)
    )

# ============================================================
# EVALUATION INSTRUCTIONS
# ============================================================

SYSTEM_PROMPT = """
You are evaluating AI-generated answers for a factual-answer hallucination study.

For every item choose exactly ONE code.

C  = Correct

E1 = Hallucinated / unsupported elaboration
The core answer is correct, but the response adds unsupported or invented factual information.

E2 = Incorrect factual / unsupported claim
The core answer is wrong OR the response contains a materially false factual claim.

E3 = No answer / unsupported-context refusal
The model refuses, says it does not know, says it cannot answer, or provides no usable answer.

E4 = Question repetition / non-answer
The model mainly repeats the question or fails to actually answer it.

Rules:
- Do not require exact wording.
- Accept correct paraphrases.
- Do not treat harmless formatting differences as errors.
- Judge the actual meaning of the answer.
- If the core answer is correct but unsupported factual additions are present, use E1.
- If the answer is materially wrong or contains a materially false factual claim, use E2.
- Return EXACTLY one code for each item.
- Preserve item order.

Return ONLY a JSON object in this exact form:

{
  "results": [
    {"item": 1, "code": "C"},
    {"item": 2, "code": "E2"}
  ]
}
"""

# ============================================================
# API EVALUATION
# ============================================================

def evaluate_batch(batch):

    prompt = SYSTEM_PROMPT + "\n\n"

    for number, (_, row) in enumerate(
        batch.iterrows(),
        start=1
    ):

        prompt += f"""
ITEM {number}

Language:
{row["Language"]}

Question:
{row["Question"]}

Gold Answer:
{row["Gold Answer"]}

Gemma Answer:
{row["Gemma Answer"]}

"""

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0,
        "max_tokens": 2500,

        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": "qa_evaluation",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "results": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "item": {
                                        "type": "integer"
                                    },
                                    "code": {
                                        "type": "string",
                                        "enum": [
                                            "C",
                                            "E1",
                                            "E2",
                                            "E3",
                                            "E4"
                                        ]
                                    }
                                },
                                "required": [
                                    "item",
                                    "code"
                                ],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["results"],
                    "additionalProperties": False
                }
            }
        }
    }

    response = requests.post(
        API_URL,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json"
        },
        json=payload,
        timeout=300
    )

    response.raise_for_status()

    data = response.json()

    message = data["choices"][0]["message"]

    content = message.get("content")

    if not content:
        raise ValueError(
            "Evaluator returned empty content."
        )

    return (
        json.loads(content),
        data.get("model", MODEL)
    )

# ============================================================
# CODE → FINAL LABEL
# ============================================================

label_map = {
    "C": (
        "Correct",
        "None"
    ),

    "E1": (
        "Error",
        "Hallucinated / unsupported elaboration"
    ),

    "E2": (
        "Error",
        "Incorrect factual / unsupported claim"
    ),

    "E3": (
        "Error",
        "No answer / unsupported-context refusal"
    ),

    "E4": (
        "Error",
        "Question repetition / non-answer"
    )
}

# ============================================================
# FIND PENDING ROWS
# ============================================================

pending = []

for idx, row in qa_results.iterrows():

    key = (
        str(row["Dataset"])
        + "|"
        + str(row["Question ID"])
        + "|"
        + str(row["Language"])
    )

    if key not in completed_keys:
        pending.append(idx)

print(
    "Pending evaluations:",
    len(pending)
)

# ============================================================
# RUN BATCHES
# ============================================================

for start in range(
    0,
    len(pending),
    BATCH_SIZE
):

    batch_indices = pending[
        start:start + BATCH_SIZE
    ]

    batch = qa_results.loc[
        batch_indices
    ].copy()

    end = start + len(batch)

    print(
        f"\nEvaluating {start + 1}-{end} "
        f"of {len(pending)}"
    )

    success = False

    # Retry a failed API call up to 3 times
    for attempt in range(1, 4):

        try:

            result, evaluator_model = evaluate_batch(
                batch
            )

            results = result["results"]

            if len(results) != len(batch):

                raise ValueError(
                    f"Expected {len(batch)} results, "
                    f"received {len(results)}"
                )

            # Check item order and codes
            for i, item in enumerate(results, start=1):

                if item.get("item") != i:
                    raise ValueError(
                        "Evaluator item numbering is incorrect."
                    )

                if item.get("code") not in {
                    "C", "E1", "E2", "E3", "E4"
                }:
                    raise ValueError(
                        f"Invalid code: {item.get('code')}"
                    )

            # ------------------------------------------------
            # Convert evaluator results into research labels
            # ------------------------------------------------

            new_rows = []

            for (_, row), item in zip(
                batch.iterrows(),
                results
            ):

                verdict, error_type = label_map[
                    item["code"]
                ]

                new_rows.append({
                    "Question ID":
                        row["Question ID"],

                    "Language":
                        row["Language"],

                    "Question":
                        row["Question"],

                    "Gold Answer":
                        row["Gold Answer"],

                    "Gemma Answer":
                        row["Gemma Answer"],

                    "Dataset":
                        row["Dataset"],

                    "Verdict":
                        verdict,

                    "Error Type":
                        error_type,

                    "Evaluator Model":
                        evaluator_model
                })

            new_df = pd.DataFrame(
                new_rows
            )

            evaluation_df = pd.concat(
                [
                    evaluation_df,
                    new_df
                ],
                ignore_index=True
            )

            # ------------------------------------------------
            # SAVE AFTER EVERY BATCH
            # ------------------------------------------------

            evaluation_df.to_csv(
                OUTPUT_FILE,
                index=False,
                encoding="utf-8-sig"
            )

            print(
                f"✅ Saved batch. "
                f"Total evaluated: {len(evaluation_df)}"
            )

            success = True
            break

        except Exception as e:

            print(
                f"Attempt {attempt}/3 failed:",
                type(e).__name__,
                str(e)
            )

            if attempt < 3:
                time.sleep(5)

    if not success:

        print(
            "\n❌ Could not evaluate this batch."
        )

        print(
            "Everything before this batch remains saved."
        )

        break

# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("Q&A EVALUATION STATUS")
print("=" * 70)

print(
    "Evaluated:",
    len(evaluation_df)
)

print(
    "Target:",
    len(qa_results)
)

print(
    "Remaining:",
    len(qa_results) - len(evaluation_df)
)

print(
    "Saved to:",
    OUTPUT_FILE
)

if len(evaluation_df) == len(qa_results):

    print("\n🎉 ALL Q&A RESPONSES EVALUATED!")

    print("\nVerdict:")
    print(
        evaluation_df["Verdict"]
        .value_counts()
    )

    print("\nError Type:")
    print(
        evaluation_df["Error Type"]
        .value_counts()
    )

Total Q&A responses: 3706
Already evaluated: 0
Pending evaluations: 3706

Evaluating 1-50 of 3706
Attempt 1/3 failed: ValueError Evaluator returned empty content.
Attempt 2/3 failed: ValueError Evaluator returned empty content.
Attempt 3/3 failed: ValueError Evaluator returned empty content.

❌ Could not evaluate this batch.
Everything before this batch remains saved.

Q&A EVALUATION STATUS
Evaluated: 0
Target: 3706
Remaining: 3706
Saved to: E:\Desktop\Research_Hallucination\Q&A_final_evaluation.csv


In [ ]:
import requests
import json

# Use the API_KEY variable that already worked.

SYSTEM_PROMPT = """
You are evaluating AI-generated factual answers.

For each item choose exactly ONE code:

C  = Correct
E1 = Hallucinated / unsupported elaboration
E2 = Incorrect factual / unsupported claim
E3 = No answer / unsupported-context refusal
E4 = Question repetition / non-answer

Judge semantic correctness, not exact wording.

Return ONLY valid JSON:
{
  "results": [
    {"item": 1, "code": "C"}
  ]
}
"""

sample = qa_results.head(5)

prompt = SYSTEM_PROMPT + "\n\n"

for i, (_, row) in enumerate(sample.iterrows(), start=1):
    prompt += f"""
ITEM {i}
Language: {row["Language"]}
Question: {row["Question"]}
Gold Answer: {row["Gold Answer"]}
Gemma Answer: {row["Gemma Answer"]}
"""

payload = {
    "model": "openai/gpt-oss-20b",
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ],
    "temperature": 0,
    "max_tokens": 2000,

    "reasoning": {
        "effort": "low",
        "exclude": True
    },

    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "qa_eval",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "results": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "item": {
                                    "type": "integer"
                                },
                                "code": {
                                    "type": "string",
                                    "enum": [
                                        "C", "E1", "E2", "E3", "E4"
                                    ]
                                }
                            },
                            "required": ["item", "code"],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["results"],
                "additionalProperties": False
            }
        }
    }
}

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    },
    json=payload,
    timeout=180
)

print("Status:", response.status_code)

if response.status_code == 200:

    data = response.json()
    message = data["choices"][0]["message"]

    print("Model:", data.get("model"))
    print("Content:", repr(message.get("content")))
    print("Reasoning present:", bool(message.get("reasoning")))
    print("Reasoning details present:",
          bool(message.get("reasoning_details")))

else:
    print(response.text)

Status: 200
Model: openai/gpt-oss-20b
Content: '{\n  "results": [\n    {"code": "C", "item": 1},\n    {"code": "C", "item": 2},\n    {"code": "C", "item": 3},\n    {"code": "E2", "item": 4},\n    {"code": "E2", "item": 5}\n  ]\n}'
Reasoning present: False
Reasoning details present: False


In [ ]:
import requests

response = requests.get(
    "https://openrouter.ai/api/v1/auth/key",
    headers={
        "Authorization": f"Bearer {API_KEY}"
    },
    timeout=30
)

print("Status:", response.status_code)
print(response.text)

Status: 200
{"data":{"label":"sk-or-v1-a2b...fd5","is_management_key":false,"is_provisioning_key":false,"limit":null,"limit_reset":null,"limit_remaining":null,"include_byok_in_limit":false,"usage":0.00140917,"usage_daily":0.00138667,"usage_weekly":0.00140917,"usage_monthly":0.00140917,"byok_usage":0,"byok_usage_daily":0,"byok_usage_weekly":0,"byok_usage_monthly":0,"is_free_tier":true,"expires_at":null,"creator_user_id":"user_3DKrB08En3IgTzGQn4gfq34ZijG","rate_limit":{"requests":-1,"interval":"10s","note":"This field is deprecated and safe to ignore."}}}


In [ ]:
# Take the first 10 Q&A responses
test_batch = qa_results.head(10).copy()

print("Testing production batch size:", len(test_batch))

# Re-use the evaluator function from the successful 5-item test,
# but send 10 real rows.

prompt = SYSTEM_PROMPT + "\n\n"

for i, (_, row) in enumerate(
    test_batch.iterrows(),
    start=1
):
    prompt += f"""
ITEM {i}
Language: {row["Language"]}
Question: {row["Question"]}
Gold Answer: {row["Gold Answer"]}
Gemma Answer: {row["Gemma Answer"]}
"""

payload = {
    "model": "openai/gpt-oss-20b",
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ],
    "temperature": 0,
    "max_tokens": 1000,
    "reasoning": {
        "effort": "low",
        "exclude": True
    },
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "qa_eval",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "results": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "item": {
                                    "type": "integer"
                                },
                                "code": {
                                    "type": "string",
                                    "enum": [
                                        "C", "E1", "E2", "E3", "E4"
                                    ]
                                }
                            },
                            "required": ["item", "code"],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["results"],
                "additionalProperties": False
            }
        }
    }
}

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    },
    json=payload,
    timeout=180
)

print("Status:", response.status_code)

if response.status_code == 200:

    data = response.json()

    print("Model:", data.get("model"))

    content = data["choices"][0]["message"].get("content")

    print("\nEvaluator output:")
    print(content)

    if content:
        parsed = json.loads(content)

        print("\nNumber of results:",
              len(parsed["results"]))

else:
    print(response.text)

Testing production batch size: 10
Status: 200
Model: openai/gpt-oss-20b

Evaluator output:
{
  "results": [
    {"item": 1, "code": "C"},
    {"item": 2, "code": "C"},
    {"item": 3, "code": "E2"},
    {"item": 4, "code": "E2"},
    {"item": 5, "code": "E2"},
    {"item": 6, "code": "E2"},
    {"item": 7, "code": "E2"},
    {"item": 8, "code": "E2"},
    {"item": 9, "code": "E2"},
    {"item": 10, "code": "E2"}
  ]
}

Number of results: 10


In [4]:
from getpass import getpass

API_KEY = getpass("Enter your OpenRouter API key: ")
print("API key loaded successfully.")

API key loaded successfully.


In [5]:
import pandas as pd
import requests
import json
import os
import time

INPUT_FOLDER = r"E:\Desktop\Research_Hallucination\gemma_results"
OUTPUT_FILE = r"E:\Desktop\Research_Hallucination\Q&A_final_evaluation.csv"

API_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "openai/gpt-oss-20b"
BATCH_SIZE = 10

# ============================================================
# LOAD RAW Q&A RESULTS
# ============================================================

files = [
    "Constitution_Law_gemma_results.csv",
    "Culture_Religion_Literature_gemma_results.csv",
    "Geography and tourism_gemma_results.csv",
    "Education health_gemma_results.csv",
    "Economy _ Technology _ Miscella_gemma_results.csv"
]

frames = []

for filename in files:
    path = os.path.join(INPUT_FOLDER, filename)
    temp = pd.read_csv(path, encoding="utf-8-sig")
    temp["Dataset"] = filename.replace("_gemma_results.csv", "")
    frames.append(temp)

qa_results = pd.concat(frames, ignore_index=True)

print("Total Q&A responses:", len(qa_results))

# ============================================================
# LOAD EXISTING EVALUATION
# ============================================================

output_columns = [
    "Question ID",
    "Language",
    "Question",
    "Gold Answer",
    "Gemma Answer",
    "Dataset",
    "Verdict",
    "Error Type",
    "Evaluator Model"
]

if os.path.exists(OUTPUT_FILE):
    try:
        evaluation_df = pd.read_csv(
            OUTPUT_FILE,
            encoding="utf-8-sig"
        )

        if not all(c in evaluation_df.columns for c in output_columns):
            evaluation_df = pd.DataFrame(columns=output_columns)

    except Exception:
        evaluation_df = pd.DataFrame(columns=output_columns)
else:
    evaluation_df = pd.DataFrame(columns=output_columns)

print("Already evaluated:", len(evaluation_df))

# ============================================================
# RESUME KEYS
# ============================================================

completed_keys = set()

if len(evaluation_df) > 0:
    completed_keys = set(
        evaluation_df["Dataset"].astype(str)
        + "|"
        + evaluation_df["Question ID"].astype(str)
        + "|"
        + evaluation_df["Language"].astype(str)
    )

# ============================================================
# EVALUATION PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are evaluating AI-generated factual answers for a research study.

For each item choose exactly one code:

C = Correct

E1 = Hallucinated / unsupported elaboration
The core answer is correct, but the response adds unsupported or invented factual information.

E2 = Incorrect factual / unsupported claim
The core answer is wrong OR the response contains a materially false factual claim.

E3 = No answer / unsupported-context refusal
The model refuses, says it does not know, cannot answer, or provides no usable answer.

E4 = Question repetition / non-answer
The model mainly repeats the question or otherwise fails to answer it.

Rules:
- Judge semantic meaning, not exact wording.
- Accept correct paraphrases.
- Harmless formatting differences are not errors.
- Extra information is not automatically an error.
- Use E1 when the core answer is correct but unsupported factual additions are present.
- Use E2 when the answer is materially wrong or contains a materially false factual claim.
- Preserve item order.
- Return ONLY the requested JSON object.

Return exactly:

{
  "results": [
    {"item": 1, "code": "C"}
  ]
}
"""

# ============================================================
# API CALL
# ============================================================

def evaluate_batch(batch):

    prompt = SYSTEM_PROMPT + "\n\n"

    for i, (_, row) in enumerate(batch.iterrows(), start=1):
        prompt += f"""
ITEM {i}

Language:
{row["Language"]}

Question:
{row["Question"]}

Gold Answer:
{row["Gold Answer"]}

Gemma Answer:
{row["Gemma Answer"]}
"""

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0,
        "max_tokens": 1000,
        "reasoning": {
            "effort": "low",
            "exclude": True
        },
        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": "qa_evaluation",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "results": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "item": {
                                        "type": "integer"
                                    },
                                    "code": {
                                        "type": "string",
                                        "enum": [
                                            "C", "E1", "E2", "E3", "E4"
                                        ]
                                    }
                                },
                                "required": [
                                    "item",
                                    "code"
                                ],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["results"],
                    "additionalProperties": False
                }
            }
        }
    }

    r = requests.post(
        API_URL,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json"
        },
        json=payload,
        timeout=180
    )

    r.raise_for_status()

    data = r.json()

    content = data["choices"][0]["message"].get("content")

    if not content:
        raise ValueError("Evaluator returned empty content.")

    return json.loads(content), data.get("model", MODEL)

# ============================================================
# LABEL MAP
# ============================================================

label_map = {
    "C": (
        "Correct",
        "None"
    ),
    "E1": (
        "Error",
        "Hallucinated / unsupported elaboration"
    ),
    "E2": (
        "Error",
        "Incorrect factual / unsupported claim"
    ),
    "E3": (
        "Error",
        "No answer / unsupported-context refusal"
    ),
    "E4": (
        "Error",
        "Question repetition / non-answer"
    )
}

# ============================================================
# FIND PENDING ROWS
# ============================================================

pending = []

for idx, row in qa_results.iterrows():

    key = (
        str(row["Dataset"])
        + "|"
        + str(row["Question ID"])
        + "|"
        + str(row["Language"])
    )

    if key not in completed_keys:
        pending.append(idx)

print("Pending evaluations:", len(pending))

# ============================================================
# RUN
# ============================================================

for start in range(0, len(pending), BATCH_SIZE):

    batch_indices = pending[start:start + BATCH_SIZE]
    batch = qa_results.loc[batch_indices].copy()

    print(
        f"\nEvaluating {start + 1}-"
        f"{start + len(batch)} of {len(pending)}"
    )

    success = False

    for attempt in range(1, 4):

        try:

            result, evaluator_model = evaluate_batch(batch)

            results = result["results"]

            if len(results) != len(batch):
                raise ValueError(
                    f"Expected {len(batch)} results, "
                    f"got {len(results)}"
                )

            new_rows = []

            for position, ((_, row), item) in enumerate(
                zip(batch.iterrows(), results),
                start=1
            ):

                if item.get("item") != position:
                    raise ValueError(
                        f"Wrong item number: {item.get('item')}"
                    )

                code = item.get("code")

                if code not in label_map:
                    raise ValueError(
                        f"Invalid code: {code}"
                    )

                verdict, error_type = label_map[code]

                new_rows.append({
                    "Question ID": row["Question ID"],
                    "Language": row["Language"],
                    "Question": row["Question"],
                    "Gold Answer": row["Gold Answer"],
                    "Gemma Answer": row["Gemma Answer"],
                    "Dataset": row["Dataset"],
                    "Verdict": verdict,
                    "Error Type": error_type,
                    "Evaluator Model": evaluator_model
                })

            evaluation_df = pd.concat(
                [
                    evaluation_df,
                    pd.DataFrame(new_rows)
                ],
                ignore_index=True
            )

            # SAVE AFTER EVERY BATCH
            evaluation_df.to_csv(
                OUTPUT_FILE,
                index=False,
                encoding="utf-8-sig"
            )

            print(
                f"✅ Batch saved | "
                f"Total evaluated: {len(evaluation_df)}"
            )

            success = True
            break

        except Exception as e:

            print(
                f"Attempt {attempt}/3 failed: "
                f"{type(e).__name__}: {e}"
            )

            if attempt < 3:
                time.sleep(5)

    if not success:

        print("\n❌ Batch failed.")
        print("Everything before this batch is saved.")
        break

print("\n" + "=" * 70)
print("FINAL STATUS")
print("=" * 70)

print("Evaluated:", len(evaluation_df))
print("Target:", len(qa_results))
print("Remaining:", len(qa_results) - len(evaluation_df))
print("Saved:", OUTPUT_FILE)

if len(evaluation_df) == len(qa_results):
    print("\n🎉 ALL 3,706 Q&A RESPONSES EVALUATED!")
    print("\nVerdict counts:")
    print(evaluation_df["Verdict"].value_counts())

    print("\nError types:")
    print(evaluation_df["Error Type"].value_counts())

Total Q&A responses: 3706
Already evaluated: 3190
Pending evaluations: 520

Evaluating 1-10 of 520
✅ Batch saved | Total evaluated: 3200

Evaluating 11-20 of 520
✅ Batch saved | Total evaluated: 3210

Evaluating 21-30 of 520
✅ Batch saved | Total evaluated: 3220

Evaluating 31-40 of 520
✅ Batch saved | Total evaluated: 3230

Evaluating 41-50 of 520
✅ Batch saved | Total evaluated: 3240

Evaluating 51-60 of 520
✅ Batch saved | Total evaluated: 3250

Evaluating 61-70 of 520
✅ Batch saved | Total evaluated: 3260

Evaluating 71-80 of 520
✅ Batch saved | Total evaluated: 3270

Evaluating 81-90 of 520
✅ Batch saved | Total evaluated: 3280

Evaluating 91-100 of 520
✅ Batch saved | Total evaluated: 3290

Evaluating 101-110 of 520
✅ Batch saved | Total evaluated: 3300

Evaluating 111-120 of 520
✅ Batch saved | Total evaluated: 3310

Evaluating 121-130 of 520
✅ Batch saved | Total evaluated: 3320

Evaluating 131-140 of 520
✅ Batch saved | Total evaluated: 3330

Evaluating 141-150 of 520
✅ Batch 

In [6]:
import pandas as pd

FILE = r"E:\Desktop\Research_Hallucination\Q&A_final_evaluation.csv"

df = pd.read_csv(FILE, encoding="utf-8-sig")

print("Rows before cleanup:", len(df))

# Identify duplicate question-language entries
key_cols = ["Dataset", "Question ID", "Language"]

duplicates = df[
    df.duplicated(
        subset=key_cols,
        keep=False
    )
].sort_values(key_cols)

print("Duplicate rows:", len(duplicates))

if len(duplicates) > 0:
    print("\nDuplicates found:")
    display(duplicates[key_cols + ["Verdict", "Error Type"]])

# Keep the first occurrence of each unique question-language pair
df_clean = df.drop_duplicates(
    subset=key_cols,
    keep="first"
).copy()

print("\nRows after cleanup:", len(df_clean))
print("Expected:", 3706)

# Save cleaned final evaluation
df_clean.to_csv(
    FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ Cleaned evaluation file saved.")

Rows before cleanup: 3710
Duplicate rows: 8

Duplicates found:


,Dataset,Question ID,Language,Verdict,Error Type
1495,Constitution_Law,NaN,English,Error,Incorrect factual / unsupported claim
3190,Constitution_Law,NaN,English,Error,No answer / unsupported-context refusal
1496,Constitution_Law,NaN,Nepali,Error,Incorrect factual / unsupported claim
3191,Constitution_Law,NaN,Nepali,Error,No answer / unsupported-context refusal
2996,Geography and tourism,NaN,English,Error,Incorrect factual / unsupported claim
3192,Geography and tourism,NaN,English,Error,No answer / unsupported-context refusal
2997,Geography and tourism,NaN,Nepali,Error,Incorrect factual / unsupported claim
3193,Geography and tourism,NaN,Nepali,Error,No answer / unsupported-context refusal



Rows after cleanup: 3706
Expected: 3706

✅ Cleaned evaluation file saved.


In [7]:
import pandas as pd
import os

FILE = r"E:\Desktop\Research_Hallucination\Q&A_final_evaluation.csv"

df = pd.read_csv(FILE, encoding="utf-8-sig")

print("=" * 70)
print("FINAL Q&A DATA CHECK")
print("=" * 70)

print("Total responses:", len(df))
print("English:", (df["Language"] == "English").sum())
print("Nepali:", (df["Language"] == "Nepali").sum())

print("\nOverall verdict:")
print(df["Verdict"].value_counts())

# ============================================================
# Overall language statistics
# ============================================================

language_rows = []

for language in ["English", "Nepali"]:

    subset = df[df["Language"] == language]

    total = len(subset)
    errors = (subset["Verdict"] == "Error").sum()
    correct = (subset["Verdict"] == "Correct").sum()

    error_rate = errors / total * 100
    accuracy = correct / total * 100

    language_rows.append({
        "Language": language,
        "Responses": total,
        "Correct": correct,
        "Errors": errors,
        "Accuracy (%)": round(accuracy, 2),
        "Error Rate (%)": round(error_rate, 2)
    })

language_summary = pd.DataFrame(language_rows)

print("\n" + "=" * 70)
print("LANGUAGE COMPARISON")
print("=" * 70)

print(language_summary.to_string(index=False))

# ============================================================
# Error-type breakdown
# ============================================================

error_df = df[df["Verdict"] == "Error"].copy()

error_summary = (
    error_df
    .groupby(["Language", "Error Type"])
    .size()
    .reset_index(name="Count")
)

error_summary["Percentage of Language Errors (%)"] = (
    error_summary.groupby("Language")["Count"]
    .transform(lambda x: x / x.sum() * 100)
).round(2)

print("\n" + "=" * 70)
print("ERROR TYPE BREAKDOWN")
print("=" * 70)

print(error_summary.to_string(index=False))

# ============================================================
# Dataset-level statistics
# ============================================================

dataset_summary = (
    df.groupby(["Dataset", "Language"])
    .agg(
        Responses=("Verdict", "size"),
        Correct=("Verdict", lambda x: (x == "Correct").sum()),
        Errors=("Verdict", lambda x: (x == "Error").sum())
    )
    .reset_index()
)

dataset_summary["Accuracy (%)"] = (
    dataset_summary["Correct"]
    / dataset_summary["Responses"]
    * 100
).round(2)

dataset_summary["Error Rate (%)"] = (
    dataset_summary["Errors"]
    / dataset_summary["Responses"]
    * 100
).round(2)

print("\n" + "=" * 70)
print("DATASET-LEVEL RESULTS")
print("=" * 70)

print(dataset_summary.to_string(index=False))

# ============================================================
# Save summary files
# ============================================================

OUTPUT_FOLDER = r"E:\Desktop\Research_Hallucination"

language_summary.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "QA_language_summary.csv"
    ),
    index=False
)

error_summary.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "QA_error_type_summary.csv"
    ),
    index=False
)

dataset_summary.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "QA_dataset_summary.csv"
    ),
    index=False
)

print("\n✅ Summary CSV files created.")

FINAL Q&A DATA CHECK
Total responses: 3706
English: 1853
Nepali: 1853

Overall verdict:
Verdict
Error      2808
Correct     898
Name: count, dtype: int64

LANGUAGE COMPARISON
Language  Responses  Correct  Errors  Accuracy (%)  Error Rate (%)
 English       1853      548    1305         29.57           70.43
  Nepali       1853      350    1503         18.89           81.11

ERROR TYPE BREAKDOWN
Language                              Error Type  Count  Percentage of Language Errors (%)
 English  Hallucinated / unsupported elaboration     18                               1.38
 English   Incorrect factual / unsupported claim   1285                              98.47
 English No answer / unsupported-context refusal      2                               0.15
  Nepali  Hallucinated / unsupported elaboration     24                               1.60
  Nepali   Incorrect factual / unsupported claim   1476                              98.20
  Nepali        Question repetition / non-answer      3 

In [8]:
import pandas as pd
import os

BASE = r"E:\Desktop\Research_Hallucination"

QA_FILE = os.path.join(
    BASE,
    "Q&A_final_evaluation.csv"
)

POLY_FILE = os.path.join(
    BASE,
    "polyfever_gemma_results.csv"
)

# ============================================================
# LOAD DATA
# ============================================================

qa = pd.read_csv(
    QA_FILE,
    encoding="utf-8-sig"
)

poly = pd.read_csv(
    POLY_FILE,
    encoding="utf-8-sig"
)

# Safety cleanup
qa = qa.drop_duplicates(
    subset=["Dataset", "Question ID", "Language"],
    keep="first"
).copy()

print("=" * 70)
print("FINAL DATASET COUNTS")
print("=" * 70)

print("Q&A:", len(qa))
print("Poly-FEVER:", len(poly))

# ============================================================
# Q&A OVERALL
# ============================================================

qa_summary = (
    qa.groupby("Language")
    .agg(
        Responses=("Verdict", "size"),
        Correct=("Verdict", lambda x: (x == "Correct").sum()),
        Errors=("Verdict", lambda x: (x == "Error").sum())
    )
    .reset_index()
)

qa_summary["Accuracy (%)"] = (
    qa_summary["Correct"]
    / qa_summary["Responses"]
    * 100
).round(2)

qa_summary["Error Rate (%)"] = (
    qa_summary["Errors"]
    / qa_summary["Responses"]
    * 100
).round(2)

print("\n" + "=" * 70)
print("Q&A — ENGLISH VS NEPALI")
print("=" * 70)
print(qa_summary.to_string(index=False))

# ============================================================
# Q&A ERROR TYPES
# ============================================================

qa_errors = qa[qa["Verdict"] == "Error"]

qa_error_types = (
    qa_errors
    .groupby(["Language", "Error Type"])
    .size()
    .reset_index(name="Count")
)

qa_error_types["Percentage of Errors (%)"] = (
    qa_error_types
    .groupby("Language")["Count"]
    .transform(
        lambda x: x / x.sum() * 100
    )
).round(2)

print("\n" + "=" * 70)
print("Q&A — ERROR TYPES")
print("=" * 70)
print(qa_error_types.to_string(index=False))

# ============================================================
# Q&A DATASET LEVEL
# ============================================================

qa_dataset = (
    qa.groupby(["Dataset", "Language"])
    .agg(
        Responses=("Verdict", "size"),
        Correct=("Verdict", lambda x: (x == "Correct").sum()),
        Errors=("Verdict", lambda x: (x == "Error").sum())
    )
    .reset_index()
)

qa_dataset["Accuracy (%)"] = (
    qa_dataset["Correct"]
    / qa_dataset["Responses"]
    * 100
).round(2)

qa_dataset["Error Rate (%)"] = (
    qa_dataset["Errors"]
    / qa_dataset["Responses"]
    * 100
).round(2)

print("\n" + "=" * 70)
print("Q&A — DATASET LEVEL")
print("=" * 70)
print(qa_dataset.to_string(index=False))

# ============================================================
# POLY-FEVER
# ============================================================

poly["Gold Label"] = (
    poly["Gold Label"]
    .astype(str)
    .str.upper()
    .str.strip()
)

poly["Gemma Prediction"] = (
    poly["Gemma Prediction"]
    .astype(str)
    .str.upper()
    .str.strip()
)

poly["Correct"] = (
    poly["Gold Label"]
    == poly["Gemma Prediction"]
)

# FALSE -> TRUE
poly["Hallucination"] = (
    (poly["Gold Label"] == "FALSE") &
    (poly["Gemma Prediction"] == "TRUE")
)

poly_summary = []

for language in ["English", "Nepali"]:

    subset = poly[
        poly["Language"] == language
    ]

    total = len(subset)
    correct = subset["Correct"].sum()

    false_cases = subset[
        subset["Gold Label"] == "FALSE"
    ]

    hallucinations = false_cases[
        false_cases["Gemma Prediction"] == "TRUE"
    ].shape[0]

    accuracy = (
        correct / total * 100
    )

    hallucination_rate = (
        hallucinations / len(false_cases) * 100
        if len(false_cases) > 0
        else 0
    )

    poly_summary.append({
        "Language": language,
        "Responses": total,
        "Correct": int(correct),
        "Accuracy (%)": round(accuracy, 2),
        "FALSE Statements": len(false_cases),
        "False→True Hallucinations": hallucinations,
        "Hallucination Rate (%)":
            round(hallucination_rate, 2)
    })

poly_summary = pd.DataFrame(
    poly_summary
)

print("\n" + "=" * 70)
print("POLY-FEVER — ENGLISH VS NEPALI")
print("=" * 70)
print(poly_summary.to_string(index=False))

# ============================================================
# LANGUAGE GAPS
# ============================================================

qa_eng = qa_summary[
    qa_summary["Language"] == "English"
].iloc[0]

qa_nep = qa_summary[
    qa_summary["Language"] == "Nepali"
].iloc[0]

poly_eng = poly_summary[
    poly_summary["Language"] == "English"
].iloc[0]

poly_nep = poly_summary[
    poly_summary["Language"] == "Nepali"
].iloc[0]

print("\n" + "=" * 70)
print("KEY LANGUAGE GAPS")
print("=" * 70)

print(
    "Q&A error-rate gap "
    "(Nepali − English):",
    round(
        qa_nep["Error Rate (%)"]
        - qa_eng["Error Rate (%)"],
        2
    ),
    "percentage points"
)

print(
    "Poly-FEVER hallucination-rate gap "
    "(English − Nepali):",
    round(
        poly_eng["Hallucination Rate (%)"]
        - poly_nep["Hallucination Rate (%)"],
        2
    ),
    "percentage points"
)

# ============================================================
# SAVE FINAL ANALYSIS FILES
# ============================================================

qa_summary.to_csv(
    os.path.join(
        BASE,
        "FINAL_QA_LANGUAGE_SUMMARY.csv"
    ),
    index=False
)

qa_error_types.to_csv(
    os.path.join(
        BASE,
        "FINAL_QA_ERROR_TYPES.csv"
    ),
    index=False
)

qa_dataset.to_csv(
    os.path.join(
        BASE,
        "FINAL_QA_DATASET_RESULTS.csv"
    ),
    index=False
)

poly_summary.to_csv(
    os.path.join(
        BASE,
        "FINAL_POLYFEVER_LANGUAGE_SUMMARY.csv"
    ),
    index=False
)

print("\n✅ FINAL ANALYSIS FILES SAVED.")

FINAL DATASET COUNTS
Q&A: 3706
Poly-FEVER: 23998

Q&A — ENGLISH VS NEPALI
Language  Responses  Correct  Errors  Accuracy (%)  Error Rate (%)
 English       1853      548    1305         29.57           70.43
  Nepali       1853      350    1503         18.89           81.11

Q&A — ERROR TYPES
Language                              Error Type  Count  Percentage of Errors (%)
 English  Hallucinated / unsupported elaboration     18                      1.38
 English   Incorrect factual / unsupported claim   1285                     98.47
 English No answer / unsupported-context refusal      2                      0.15
  Nepali  Hallucinated / unsupported elaboration     24                      1.60
  Nepali   Incorrect factual / unsupported claim   1476                     98.20
  Nepali        Question repetition / non-answer      3                      0.20

Q&A — DATASET LEVEL
                        Dataset Language  Responses  Correct  Errors  Accuracy (%)  Error Rate (%)
            